# Installation

In [ ]:
!pip install -q llmcompressor datasets transformers accelerate

# Load Hugging Face Token

In [ ]:
import os

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN is None:
    print("Token not found!")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded successfully.")

# AP-Quantization 

In [2]:
# -*- coding: utf-8 -*-

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import numpy as np
import pickle
from tqdm import tqdm
from collections import defaultdict

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

SEED = 42
set_seed(SEED)

print("=" * 70)
print(f"AP-QUANT: JOINT CALIBRATION (ALL LAYERS, FIXED 4-BIT, DYNAMIC λ, QKV+Wo FULL ATTENTION, GPT2+LLaMA)")
print(f"SEED: {SEED}")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Config:
    # Model configuration
    model_name: str = "openai-community/gpt2"
    
    # Quantization configuration
    target_bits: int = 4
    candidate_bits: List[int] = field(default_factory=lambda: [2, 3, 4, 8, 16])
    
    # Dataset configuration
    dataset_name: str = "garage-bAInd/Open-Platypus"  # Options: "garage-bAInd/Open-Platypus", "Salesforce/wikitext", "allenai/c4"
    dataset_split: str = "train"
    text_column: str = "instruction"  # "instruction" for Open-Platypus, "text" for wikitext/c4
    calib_samples: int = 256
    eval_samples: int = 256  # For perplexity evaluation
    
    # Calibration parameters
    chunk_size: int = 128
    max_len_per_example: int = 256
    batch_size: int = 8  # For batching tokens
    
    # Sequential calibration parameters
    num_steps_per_layer: int = 200
    learning_rate: float = 1e-3
    init_temp: float = 2.0
    lambda_update_every: int = 10
    val_fraction: float = 0.1
    calibration_passes: int = 3
    
    # Joint calibration parameters
    num_steps_joint: int = 300
    
    # Sensitivity analysis parameters
    lambda_samples: int = 8
    
    # Random seed
    seed: int = 42
    
    # Output files
    scales_file: str = "calibration_scales_sequential.pkl"


def load_calibration_dataset(cfg: Config, tokenizer):
    """Load and prepare calibration/evaluation datasets."""
    print(f"\nLoading dataset: {cfg.dataset_name}...")
    
    if cfg.dataset_name == "garage-bAInd/Open-Platypus":
        ds = load_dataset(cfg.dataset_name, split=cfg.dataset_split)
        text_col = "instruction"
    elif cfg.dataset_name == "Salesforce/wikitext":
        ds = load_dataset(cfg.dataset_name, "wikitext-2-raw-v1", split=cfg.dataset_split)
        text_col = "text"
    elif cfg.dataset_name == "allenai/c4":
        ds = load_dataset("brando/small-c4-dataset", split=cfg.dataset_split)
        text_col = "text"
    else:
        raise ValueError(f"Unsupported dataset: {cfg.dataset_name}")
        
    # ds = ds.shuffle(seed=cfg.seed)
    # ds = ds.filter(lambda x: len(x.get(text_col, "").strip()) > 0)
    
    calib_texts = [ds[i][text_col] for i in range(min(cfg.calib_samples, len(ds)))]
    eval_start = cfg.calib_samples
    eval_end = min(eval_start + cfg.eval_samples, len(ds))
    eval_texts = [ds[i][text_col] for i in range(eval_start, eval_end)]
    
    calib_tokens = prepare_tokens(calib_texts, tokenizer, cfg.chunk_size, cfg.max_len_per_example)
    eval_tokens = prepare_tokens(eval_texts, tokenizer, cfg.chunk_size, cfg.max_len_per_example)
    
    return calib_tokens, eval_tokens
    
# ============================================================================
# 0. Exception for early-exit forward pass
# ============================================================================

class _StopForward(Exception):
    pass

# ============================================================================
# 1. Helper functions for precision awareness
# ============================================================================

def get_model_precision(model):
    """Detect the dtype of the model's parameters."""
    return next(model.parameters()).dtype

def should_use_amp(model):
    """Determine if AMP should be used based on model precision."""
    dtype = get_model_precision(model)
    return dtype in [torch.bfloat16, torch.float16]

def get_precision_tolerance(model):
    """Get appropriate tolerance for sanity checks based on model precision."""
    dtype = get_model_precision(model)
    if dtype == torch.bfloat16:
        return 0.02, 0.05  # rtol, atol for BF16
    elif dtype == torch.float16:
        return 0.01, 0.01  # rtol, atol for FP16
    else:
        return 1e-5, 1e-4  # rtol, atol for FP32

# ============================================================================
# 2. QuantizedLinear — precision-aware
# ============================================================================

class STEQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight, scale, q_max):
        scale_c = torch.clamp(scale, min=1e-8)
        w_scaled = weight / scale_c
        w_clamped = torch.clamp(w_scaled, -q_max, q_max)
        w_round = torch.round(w_clamped)
        w_quant = w_round * scale_c

        ctx.save_for_backward(w_scaled, w_clamped, w_round, scale_c)
        ctx.q_max = q_max
        return w_quant

    @staticmethod
    def backward(ctx, grad_output):
        w_scaled, w_clamped, w_round, scale_c = ctx.saved_tensors
        q_max = ctx.q_max

        not_clipped = (w_scaled.abs() <= q_max).float()
        grad_weight = grad_output * not_clipped

        grad_scale_elem = torch.where(
            w_scaled > q_max, torch.full_like(w_scaled, float(q_max)),
            torch.where(
                w_scaled < -q_max, torch.full_like(w_scaled, -float(q_max)),
                w_round - w_scaled
            )
        )
        contrib = grad_output * grad_scale_elem
        grad_scale = contrib.sum(dim=0, keepdim=True)

        return grad_weight, grad_scale, None


class QuantizedLinear(nn.Module):
    def __init__(self, weight: torch.Tensor, bias: torch.Tensor = None, bits: int = 4):
        super().__init__()
        # Store weight in its native dtype
        self.register_buffer("weight_fp", weight.clone().detach())
        if bias is not None:
            self.register_buffer("bias_fp", bias.clone().detach())
        else:
            self.bias_fp = None

        self.bits = bits
        self.q_max = 2 ** (bits - 1) - 1
        self.force_fp = False

        # Always compute scale in FP32 for numerical stability
        init_scale = (
            weight.detach().float().abs().amax(dim=0, keepdim=True) / self.q_max
        ).clamp(min=1e-6)
        init_raw = torch.log(torch.expm1(init_scale).clamp_min(1e-8))
        self.raw_scale = nn.Parameter(init_raw.float())

    @property
    def scale(self):
        return F.softplus(self.raw_scale) + 1e-8

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig_dtype = x.dtype
        
        if self.force_fp:
            # FP mode: use weight_fp directly
            w = self.weight_fp
            # Match the activation dtype for consistency
            if orig_dtype in [torch.bfloat16, torch.float16]:
                w = w.to(orig_dtype)
                out = torch.matmul(x, w)
                if self.bias_fp is not None:
                    out = out + self.bias_fp.to(orig_dtype)
                return out
            else:
                out = torch.matmul(x, w)
                if self.bias_fp is not None:
                    out = out + self.bias_fp
                return out
        else:
            # Quantized path: compute in FP32 for numerical stability
            w = STEQuantize.apply(self.weight_fp, self.scale, self.q_max)
            
            if orig_dtype in [torch.bfloat16, torch.float16]:
                # Promote to FP32 for computation
                x_fp32 = x.float()
                w_fp32 = w.float()
                out = torch.matmul(x_fp32, w_fp32)
                if self.bias_fp is not None:
                    out = out + self.bias_fp.float()
                # Cast back to original dtype
                return out.to(orig_dtype)
            else:
                # FP32 path (no casting needed)
                out = torch.matmul(x, w)
                if self.bias_fp is not None:
                    out = out + self.bias_fp
                return out

    def set_bits(self, bits: int):
        self.bits = bits
        self.q_max = 2 ** (bits - 1) - 1
        # Always compute in FP32 for stability
        init_scale = (
            self.weight_fp.float().abs().amax(dim=0, keepdim=True) / self.q_max
        ).clamp(min=1e-6)
        init_raw = torch.log(torch.expm1(init_scale).clamp_min(1e-8))
        self.raw_scale.data = init_raw.float()


# ============================================================================
# 3. GPT-2: fused QKV wrapper + native attention
# ============================================================================

class _FusedQKV(nn.Module):
    def __init__(self, q_proj: QuantizedLinear, k_proj: QuantizedLinear, v_proj: QuantizedLinear):
        super().__init__()
        self.q_proj = q_proj
        self.k_proj = k_proj
        self.v_proj = v_proj

    def forward(self, x):
        return torch.cat([self.q_proj(x), self.k_proj(x), self.v_proj(x)], dim=-1)


class NativeQuantizedGPT2Attention(nn.Module):
    def __init__(self, original_attn: nn.Module, bits: int = 4):
        super().__init__()
        self.force_fp_mode = False
        self.original_attn = original_attn

        W = original_attn.c_attn.weight.data
        b = original_attn.c_attn.bias.data if original_attn.c_attn.bias is not None else None
        w_q, w_k, w_v = W.chunk(3, dim=-1)
        b_q, b_k, b_v = b.chunk(3, dim=-1) if b is not None else (None, None, None)

        self.q_proj = QuantizedLinear(w_q, b_q, bits=bits)
        self.k_proj = QuantizedLinear(w_k, b_k, bits=bits)
        self.v_proj = QuantizedLinear(w_v, b_v, bits=bits)

        self.original_attn.c_attn = _FusedQKV(self.q_proj, self.k_proj, self.v_proj)

        # --- NEW: quantize Wo for GPT-2 ---
        W_o = original_attn.c_proj.weight.data
        b_o = original_attn.c_proj.bias.data if original_attn.c_proj.bias is not None else None
        self.o_proj = QuantizedLinear(W_o, b_o, bits=bits)

        # Patch original module
        original_attn.c_proj = self.o_proj

        # Remove the _capture_o hook entirely - we now use o_proj directly

    def set_bits(self, bits: int):
        for p in (self.q_proj, self.k_proj, self.v_proj, self.o_proj):
            p.set_bits(bits)

    def _set_force_fp(self, use_quant: bool):
        for p in (self.q_proj, self.k_proj, self.v_proj, self.o_proj):
            p.force_fp = not use_quant

    def forward_components(self, hidden_states: torch.Tensor, use_quant: bool = True, position_embeddings=None):
        self._set_force_fp(use_quant)

        out = self.original_attn(hidden_states, use_cache=False, output_attentions=True)
        
        if len(out) == 3:
            attn_output, _, P = out
        else:
            attn_output, P = out
        
        # NEW: Use quantized Wo output directly
        # O_merged = self.o_proj(hidden_states)
        O_merged = attn_output

        T = hidden_states.shape[1]
        causal_mask = torch.triu(
            torch.ones((T, T), device=hidden_states.device, dtype=torch.bool), diagonal=1
        )
        return P, O_merged, causal_mask

    def forward(self, hidden_states, **kwargs):
        self._set_force_fp(use_quant=not self.force_fp_mode)
        return self.original_attn(hidden_states, **kwargs)


# ============================================================================
# 4. LLaMA: use native forward (no hand-rolled RoPE/GQA)
# ============================================================================

class NativeQuantizedLlamaAttention(nn.Module):
    def __init__(self, original_attn, bits: int = 4):
        super().__init__()
        self.force_fp_mode = False
        self.original_attn = original_attn

        W_q = original_attn.q_proj.weight.data.T.contiguous()
        b_q = original_attn.q_proj.bias.data if original_attn.q_proj.bias is not None else None
        W_k = original_attn.k_proj.weight.data.T.contiguous()
        b_k = original_attn.k_proj.bias.data if original_attn.k_proj.bias is not None else None
        W_v = original_attn.v_proj.weight.data.T.contiguous()
        b_v = original_attn.v_proj.bias.data if original_attn.v_proj.bias is not None else None

        self.q_proj = QuantizedLinear(W_q, b_q, bits=bits)
        self.k_proj = QuantizedLinear(W_k, b_k, bits=bits)
        self.v_proj = QuantizedLinear(W_v, b_v, bits=bits)

        self.original_attn.q_proj = self.q_proj
        self.original_attn.k_proj = self.k_proj
        self.original_attn.v_proj = self.v_proj

        # --- NEW: quantize Wo for LLaMA ---
        W_o = original_attn.o_proj.weight.data.T.contiguous()
        b_o = original_attn.o_proj.bias.data if original_attn.o_proj.bias is not None else None
        self.o_proj = QuantizedLinear(W_o, b_o, bits=bits)

        original_attn.o_proj = self.o_proj

        # Remove the _capture_o hook entirely

    def set_bits(self, bits: int):
        for p in (self.q_proj, self.k_proj, self.v_proj, self.o_proj):
            p.set_bits(bits)

    def _set_force_fp(self, use_quant: bool):
        for p in (self.q_proj, self.k_proj, self.v_proj, self.o_proj):
            p.force_fp = not use_quant

    def forward_components(self, hidden_states, use_quant=True,
                            position_embeddings=None, attention_mask=None):
        if position_embeddings is None:
            raise ValueError("position_embeddings=(cos, sin) is required")

        self._set_force_fp(use_quant)

        T = hidden_states.shape[1]
        if attention_mask is None:
            attention_mask = build_causal_mask(T, hidden_states.device, hidden_states.dtype)

        out = self.original_attn(
            hidden_states,
            attention_mask=attention_mask,
            position_embeddings=position_embeddings,
            output_attentions=True,
        )
        
        if len(out) >= 2:
            attn_output, P = out[0], out[1]
        else:
            attn_output = out[0]
            P = None
            
        # NEW: Use quantized Wo output directly
        # O_merged = self.o_proj(hidden_states)
        O_merged = attn_output

        causal_mask = torch.triu(
            torch.ones((T, T), device=hidden_states.device, dtype=torch.bool), diagonal=1
        )
        return P, O_merged, causal_mask

    def forward(self, hidden_states, attention_mask=None, position_ids=None,
                past_key_value=None, output_attentions=False,
                position_embeddings=None, **kwargs):
        self._set_force_fp(use_quant=not self.force_fp_mode)
        return self.original_attn(
            hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_value=past_key_value,
            output_attentions=output_attentions,
            position_embeddings=position_embeddings,
            **kwargs,
        )


def build_causal_mask(T, device, dtype):
    mask_bool = torch.triu(torch.ones((T, T), device=device, dtype=torch.bool), diagonal=1)
    mask = torch.zeros((T, T), device=device, dtype=dtype)
    mask.masked_fill_(mask_bool, torch.finfo(dtype).min)
    return mask.unsqueeze(0).unsqueeze(0)


def get_position_embeddings(model, hidden_states):
    B, T, _ = hidden_states.shape
    position_ids = torch.arange(T, device=hidden_states.device).unsqueeze(0).expand(B, -1)
    return model.model.rotary_emb(hidden_states, position_ids)


# ============================================================================
# 5. Detection and quantization functions
# ============================================================================

def detect_arch(model):
    if hasattr(model, "transformer"):
        return "gpt2"
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return "llama"
    raise ValueError("Unsupported model architecture")


def quantize_transformer_attn(model, layer_bits=None):
    arch = detect_arch(model)
    q_attn_blocks = []
    
    if arch == "gpt2":
        for layer_idx, block in enumerate(model.transformer.h):
            bits = layer_bits[layer_idx] if layer_bits is not None else 4
            q_attn = NativeQuantizedGPT2Attention(block.attn, bits=bits).to(device)
            block.attn = q_attn
            q_attn_blocks.append(q_attn)
            print(f"  GPT2 layer {layer_idx:02d}: attention quantized to {bits}-bit (QKV+Wo FULL ATTENTION)")
    elif arch == "llama":
        for layer_idx, block in enumerate(model.model.layers):
            bits = layer_bits[layer_idx] if layer_bits is not None else 4
            q_attn = NativeQuantizedLlamaAttention(block.self_attn, bits=bits).to(device)
            block.self_attn = q_attn
            q_attn_blocks.append(q_attn)
            print(f"  LLaMA layer {layer_idx:02d}: attention quantized to {bits}-bit (QKV+Wo FULL ATTENTION)")
    else:
        raise ValueError(f"Unsupported arch: {arch}")
    
    return arch, q_attn_blocks


# ============================================================================
# 6. Collect layer inputs - OPTIMIZED with early-exit and batching
# ============================================================================

def collect_layer_inputs(model, arch: str, calib_tokens, device, max_layer_idx=None, use_amp=None):
    """Collect layer inputs with early-exit and optional AMP for speed.
    Auto-detects AMP based on model precision if use_amp is None."""
    
    if arch == "gpt2":
        num_layers = len(model.transformer.h)
    elif arch == "llama":
        num_layers = len(model.model.layers)
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    # Auto-detect AMP if not specified
    if use_amp is None:
        use_amp = should_use_amp(model) and device == "cuda"

    layer_inputs = [[] for _ in range(num_layers)]
    
    # Determine where to stop
    stop_at = num_layers - 1 if max_layer_idx is None else max_layer_idx

    def make_hook(layer_idx):
        def hook_fn(module, inp, out):
            # Store the output (hidden state) for this layer
            # Split batch into individual samples for storage
            for row in out.detach().split(1, dim=0):
                layer_inputs[layer_idx].append(row.cpu())  # Move to CPU to save GPU memory
            # If we've reached the target layer, stop the forward pass
            if layer_idx == stop_at:
                raise _StopForward()
        return hook_fn

    handles = []
    if arch == "gpt2":
        for layer_idx, block in enumerate(model.transformer.h):
            h = block.ln_1.register_forward_hook(make_hook(layer_idx))
            handles.append(h)
    else:
        for layer_idx, block in enumerate(model.model.layers):
            h = block.input_layernorm.register_forward_hook(make_hook(layer_idx))
            handles.append(h)

    with torch.no_grad():
        if use_amp and device == "cuda":
            # Use the model's native dtype for AMP
            model_dtype = get_model_precision(model)
            with torch.autocast(device_type="cuda", dtype=model_dtype):
                for chunk in calib_tokens:
                    try:
                        model(chunk.to(device), use_cache=False)
                    except _StopForward:
                        pass
        else:
            for chunk in calib_tokens:
                try:
                    model(chunk.to(device), use_cache=False)
                except _StopForward:
                    pass

    for h in handles:
        h.remove()

    return layer_inputs


# ============================================================================
# 6b. Batch tokens helper
# ============================================================================

def batch_tokens(calib_tokens, batch_size=8):
    """Group tokens by length and create batches for efficient GPU utilization."""
    groups = defaultdict(list)
    for t in calib_tokens:
        groups[t.shape[1]].append(t)
    
    batches = []
    for length, chunks in groups.items():
        for i in range(0, len(chunks), batch_size):
            batches.append(torch.cat(chunks[i:i+batch_size], dim=0))
    return batches


# ============================================================================
# 7. compute_ap_loss_improved
# ============================================================================

def compute_ap_loss_improved(P_fp, O_fp, P_q, O_q, causal_mask,
                         lam=1.0, temp=2.0, eps=1e-8):
    diff = O_fp - O_q
    mse_raw = diff.pow(2).mean()

    l_output = mse_raw / (O_fp.pow(2).mean() + eps)

    P_fp_c = F.softmax(torch.log(P_fp.clamp(min=eps)) / temp, dim=-1)
    P_q_c = F.softmax(torch.log(P_q.clamp(min=eps)) / temp, dim=-1)

    kl_matrix = P_fp_c * (torch.log(P_fp_c + eps) - torch.log(P_q_c + eps))

    valid_mask = (~causal_mask).unsqueeze(0).unsqueeze(0)
    kl_valid = kl_matrix.masked_fill(~valid_mask, 0.0)

    kl_per_query = kl_valid.sum(dim=-1)
    query_valid = valid_mask.any(dim=-1).to(kl_per_query.dtype)
    l_kl = (kl_per_query * query_valid).sum() / query_valid.sum().clamp_min(1.0)

    l_joint = l_output + lam * l_kl
    return l_joint, l_output, l_kl, mse_raw


def joint_loss_for_sensitivity(P1, O1, P2, O2, lam, eps=1e-6):
    """loss function for sensitivity only."""
    L_out = (O2 - O1).pow(2).sum() / (O1.pow(2).sum() + eps)
    p1 = P1.clamp(min=1e-9)
    p2 = P2.clamp(min=1e-9)
    L_kl = (p1 * (p1.log() - p2.log())).sum(dim=-1).mean()
    return L_out + lam * L_kl, L_out.item(), L_kl.item()

# ============================================================================
# 8. compute_sensitivity_matrix
# ============================================================================

@torch.no_grad()
def compute_sensitivity_matrix(
    model,
    calib_tokens,
    candidate_bits,
    temp: float = 2.0,
    device: str = "cuda",
    lambda_samples: int = 8,
):
    arch, q_attn_blocks = quantize_transformer_attn(model, layer_bits=None)

    # Force FP mode for all layers
    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = True

    # Collect FP inputs (no quantization active)
    layer_inputs = collect_layer_inputs(model, arch, calib_tokens, device)

    # Turn quantization back on
    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = False

    lam_values = []
    valid_layers = [i for i in range(len(q_attn_blocks)) if len(layer_inputs[i]) > 0]
    if not valid_layers:
        raise RuntimeError("No layer inputs collected; cannot estimate lambda.")

    num_layers_to_use = min(3, len(valid_layers))
    layers_to_use = valid_layers[:num_layers_to_use]

    for layer_idx in layers_to_use:
        q_attn = q_attn_blocks[layer_idx]

        q_attn.q_proj.set_bits(4)
        q_attn.k_proj.set_bits(4)
        q_attn.v_proj.set_bits(4)
        q_attn.o_proj.set_bits(4)

        n_samples = min(lambda_samples, len(layer_inputs[layer_idx]))
        sample_indices = np.random.choice(len(layer_inputs[layer_idx]), n_samples, replace=False)

        for idx in sample_indices:
            inp = layer_inputs[layer_idx][idx].to(device)

            if arch == "llama":
                pos_emb = get_position_embeddings(model, inp)
            else:
                pos_emb = None

            q_attn.force_fp_mode = True
            P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False, position_embeddings=pos_emb)

            q_attn.force_fp_mode = False
            P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True, position_embeddings=pos_emb)

            _, l_out_0, l_kl_0, _ = compute_ap_loss_improved(
                P_fp, O_fp, P_q, O_q, mask,
                lam=1.0,
                temp=temp,
            )

            lam_values.append(float(l_out_0 / (l_kl_0 + 1e-8)))

    lam = np.mean(lam_values) if lam_values else 1.0
    lam = min(max(lam, 0.01), 10.0)
    print(f"[Sensitivity] λ = {lam:.4f}")

    sensitivity = {i: {} for i in range(len(q_attn_blocks))}

    for layer_idx, q_attn in enumerate(q_attn_blocks):
        if len(layer_inputs[layer_idx]) == 0:
            continue

        print(f"layer {layer_idx:02d}:", end=" ")

        for bits in candidate_bits:
            q_attn.q_proj.set_bits(bits)
            q_attn.k_proj.set_bits(bits)
            q_attn.v_proj.set_bits(bits)
            q_attn.o_proj.set_bits(bits)

            total_l_out = 0.0
            total_l_kl = 0.0
            total_l_joint = 0.0
            total_mse_raw = 0.0
            n_samples = len(layer_inputs[layer_idx])

            for inp in layer_inputs[layer_idx]:
                inp = inp.to(device)

                if arch == "llama":
                    pos_emb = get_position_embeddings(model, inp)
                else:
                    pos_emb = None

                q_attn.force_fp_mode = True
                P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False, position_embeddings=pos_emb)

                q_attn.force_fp_mode = False
                P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True, position_embeddings=pos_emb)

                l_joint, l_out, l_kl, mse_raw = compute_ap_loss_improved(
                    P_fp, O_fp, P_q, O_q, mask,
                    lam=lam,
                    temp=temp,
                )
                
                total_l_joint += l_joint.item()
                total_l_out += l_out.item()
                total_l_kl += l_kl.item()
                total_mse_raw += mse_raw.item()
                
            avg_l_out = total_l_out / max(n_samples, 1)
            avg_l_kl = total_l_kl / max(n_samples, 1)
            avg_l_joint = total_l_joint / max(n_samples, 1)
            avg_mse_raw = total_mse_raw / max(n_samples, 1)
            
            sensitivity[layer_idx][bits] = avg_l_joint

            print(f"{bits}b={avg_l_joint:.4f}", end=", ")

        print()

    return sensitivity, lam


# ============================================================================
# 9. solve_mckp
# ============================================================================

def solve_mckp(sensitivity, cost, budget, candidate_bits):
    budget = int(budget)
    n = len(sensitivity)
    INF = float("inf")
    dp = [INF] * (budget + 1)
    dp[0] = 0.0
    choice = [[None] * (budget + 1) for _ in range(n)]

    for i in range(n):
        new_dp = [INF] * (budget + 1)
        for c in range(budget + 1):
            if dp[c] == INF:
                continue
            for b in candidate_bits:
                c2 = c + cost[b]
                if c2 <= budget and dp[c] + sensitivity[i][b] < new_dp[c2]:
                    new_dp[c2] = dp[c] + sensitivity[i][b]
                    choice[i][c2] = (b, c)
        dp = new_dp

    best_c = min(range(budget + 1), key=lambda c: dp[c])
    assignment, c = {}, best_c
    for i in reversed(range(n)):
        b, prev_c = choice[i][c]
        assignment[i] = b
        c = prev_c
    return assignment, dp[best_c]


def solve_mckp_advanced(
    sensitivity, 
    cost_model='log',  # 'linear', 'log', 'sqrt', 'parameter_aware'
    budget=None,
    candidate_bits=None,
    model=None,
    n_layers=None,
    lambda_multiplier=1.0,
):
    """
    Enhanced knapsack solver with multiple cost models.
    """
    n = len(sensitivity)
    if candidate_bits is None:
        candidate_bits = sorted(sensitivity[0].keys())
    
    # Choose cost model
    if cost_model == 'linear':
        cost = {b: b for b in candidate_bits}
    elif cost_model == 'log':
        cost = {b: math.log2(b + 1) * 2 for b in candidate_bits}
    elif cost_model == 'sqrt':
        cost = {b: math.sqrt(b * 2) for b in candidate_bits}
    elif cost_model == 'sqrt_aggressive':
        cost = {b: math.sqrt(b) * 1.5 for b in candidate_bits}
    elif cost_model == 'parameter_aware' and model is not None:
        # Compute per-layer costs based on parameter count
        arch = detect_arch(model)
        avg_size = 0
        layer_sizes = []
        
        for layer_idx in range(n_layers):
            if arch == "gpt2":
                block = model.transformer.h[layer_idx]
            else:
                block = model.model.layers[layer_idx]
            
            size = sum(p.numel() for p in block.parameters())
            layer_sizes.append(size)
        
        avg_size = np.mean(layer_sizes)
        cost = {}
        for layer_idx in range(n):
            normalized_size = layer_sizes[layer_idx] / avg_size
            cost[layer_idx] = {b: math.sqrt(b * 2) * normalized_size for b in candidate_bits}
    else:
        # Default: sqrt
        cost = {b: math.sqrt(b * 2) for b in candidate_bits}
    
    # If budget is None, use average bits = 4
    if budget is None:
        if isinstance(cost, dict) and all(isinstance(v, dict) for v in cost.values()):
            # Parameter aware: use average of sqrt costs
            avg_cost_per_bit = np.mean([
                cost[layer_idx][4] for layer_idx in range(n)
            ])
            budget = int(avg_cost_per_bit * n)
        else:
            avg_cost = np.mean([cost[4] for b in candidate_bits])  # Should be ~cost[4]
            budget = int(avg_cost * n)
    
    print(f"Budget: {budget} (cost model: {cost_model})")
    
    # Prepare sensitivity values
    if isinstance(cost, dict) and all(isinstance(v, dict) for v in cost.values()):
        # Parameter-aware cost
        INF = float("inf")
        dp = [INF] * (budget + 1)
        dp[0] = 0.0
        choice = [[None] * (budget + 1) for _ in range(n)]
        
        for i in range(n):
            new_dp = [INF] * (budget + 1)
            for c in range(budget + 1):
                if dp[c] == INF:
                    continue
                for b in candidate_bits:
                    c2 = c + cost[i][b]
                    if c2 <= budget and dp[c] + sensitivity[i][b] < new_dp[c2]:
                        new_dp[c2] = dp[c] + sensitivity[i][b]
                        choice[i][c2] = (b, c)
            dp = new_dp
        
        best_c = min(range(budget + 1), key=lambda c: dp[c])
        assignment, c = {}, best_c
        for i in reversed(range(n)):
            b, prev_c = choice[i][c]
            assignment[i] = b
            c = prev_c
        total_loss = dp[best_c]
    else:
        # Single cost model
        INF = float("inf")
        dp = [INF] * (budget + 1)
        dp[0] = 0.0
        choice = [[None] * (budget + 1) for _ in range(n)]
        
        for i in range(n):
            new_dp = [INF] * (budget + 1)
            for c in range(budget + 1):
                if dp[c] == INF:
                    continue
                for b in candidate_bits:
                    c2 = c + cost[b]
                    if c2 <= budget and dp[c] + sensitivity[i][b] < new_dp[c2]:
                        new_dp[c2] = dp[c] + sensitivity[i][b]
                        choice[i][c2] = (b, c)
            dp = new_dp
        
        best_c = min(range(budget + 1), key=lambda c: dp[c])
        assignment, c = {}, best_c
        for i in reversed(range(n)):
            b, prev_c = choice[i][c]
            assignment[i] = b
            c = prev_c
        total_loss = dp[best_c]
    
    return assignment, total_loss, budget
    
# ============================================================================
# 10. calibrate_ap_quant_sequential - OPTIMIZED with early-exit and batching
# ============================================================================

def calibrate_ap_quant_sequential(
    model,
    calib_tokens,
    device,
    layer_bits=None,
    num_steps_per_layer=200,
    lr=1e-3,
    init_temp=2.0,
    lambda_update_every=10,
    batch_size=4,
    val_fraction=0.1,
    calibration_passes=2,
):
    model.eval()
    model.to(device)

    arch, q_attn_blocks = quantize_transformer_attn(model, layer_bits=layer_bits)

    print("\n" + "=" * 70)
    print(f"AP-QUANT: SEQUENTIAL CALIBRATION ({arch.upper()}, 4-BIT, DYNAMIC λ, QKV+Wo FULL ATTENTION)")
    print(f"Steps/layer: {num_steps_per_layer}, Batch size: {batch_size}, Passes: {calibration_passes}")
    print("Full validation set checkpointing: ENABLED")
    print("=" * 70)

    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = True

    layer_inputs_fp = collect_layer_inputs(
        model, arch=arch, calib_tokens=calib_tokens, device=device,
        max_layer_idx=None,
        use_amp=None
    )

    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = False

    print("\n  Initializing scales with RTN...")
    for layer_idx, q_attn in enumerate(q_attn_blocks):
        if layer_bits is not None and layer_idx in layer_bits:
            bits = layer_bits[layer_idx]
            q_attn.q_proj.set_bits(bits)
            q_attn.k_proj.set_bits(bits)
            q_attn.v_proj.set_bits(bits)
            q_attn.o_proj.set_bits(bits)
        else:
            q_attn.q_proj.set_bits(4)
            q_attn.k_proj.set_bits(4)
            q_attn.v_proj.set_bits(4)
            q_attn.o_proj.set_bits(4)
        
        q_attn.force_fp_mode = False

    print(f"  raw_scale dtypes after initialization:")
    for layer_idx, q_attn in enumerate(q_attn_blocks):
        print(f"    Layer {layer_idx:02d}: q={q_attn.q_proj.raw_scale.dtype}, k={q_attn.k_proj.raw_scale.dtype}, v={q_attn.v_proj.raw_scale.dtype}, o={q_attn.o_proj.raw_scale.dtype}")

    best_scales_overall = [(
        q_attn.q_proj.raw_scale.data.clone(),
        q_attn.k_proj.raw_scale.data.clone(),
        q_attn.v_proj.raw_scale.data.clone(),
        q_attn.o_proj.raw_scale.data.clone(),
    ) for q_attn in q_attn_blocks]
    
    best_overall_loss = float('inf')

    for pass_idx in range(calibration_passes):
        print(f"\n{'='*70}")
        print(f"SEQUENTIAL CALIBRATION PASS {pass_idx + 1}/{calibration_passes}")
        print(f"{'='*70}")

        for layer_idx, q_attn in enumerate(q_attn_blocks):
            if len(layer_inputs_fp[layer_idx]) == 0:
                print(f"  Layer {layer_idx:02d}: no inputs collected, skipping.")
                continue

            print(f"\n--- Calibrating layer {layer_idx:02d} (pass {pass_idx + 1}) ---")

            q_attn.force_fp_mode = True

            layer_inputs_quant = collect_layer_inputs(
                model, arch=arch, calib_tokens=calib_tokens, device=device,
                max_layer_idx=layer_idx,
                use_amp=None
            )
            q_attn.force_fp_mode = False

            current_inputs = layer_inputs_quant[layer_idx]
            
            if len(current_inputs) == 0:
                print(f"  Layer {layer_idx:02d}: no quantized inputs collected, using FP inputs.")
                current_inputs = layer_inputs_fp[layer_idx]

            n_samples = len(current_inputs)
            n_val = max(1, int(n_samples * val_fraction))
            n_train = n_samples - n_val
            
            shuffled_indices = np.random.permutation(n_samples)
            train_indices = shuffled_indices[:n_train]
            val_indices = shuffled_indices[n_train:]
            
            train_inputs = [current_inputs[i] for i in train_indices]
            val_inputs = [current_inputs[i] for i in val_indices]
            
            print(f"  Layer {layer_idx:02d}: {n_train} train, {n_val} val samples")

            params = [
                q_attn.q_proj.raw_scale,
                q_attn.k_proj.raw_scale,
                q_attn.v_proj.raw_scale,
                q_attn.o_proj.raw_scale,   # NEW: Add Wo to calibration
            ]

            q_attn.force_fp_mode = False

            optimizer = torch.optim.Adam(params, lr=lr)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps_per_layer)

            lam_values = []
            n_lambda_samples = min(8, len(train_inputs))
            if n_lambda_samples > 0:
                sample_indices = np.random.choice(len(train_inputs), n_lambda_samples, replace=False)
                
                for idx in sample_indices:
                    sample_inp = train_inputs[idx].to(device)
                    
                    if arch == "llama":
                        pos_emb = get_position_embeddings(model, sample_inp)
                    else:
                        pos_emb = None
                    
                    q_attn.force_fp_mode = True
                    P_fp, O_fp, mask = q_attn.forward_components(sample_inp, use_quant=False, position_embeddings=pos_emb)
                    q_attn.force_fp_mode = False
                    P_q, O_q, _ = q_attn.forward_components(sample_inp, use_quant=True, position_embeddings=pos_emb)
                    _, l_out_0, l_kl_0, _ = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=1.0, temp=init_temp
                    )
                    lam_val = float(l_out_0 / (l_kl_0 + 1e-8))
                    lam_values.append(lam_val)
            
            lam_val = min(max(np.mean(lam_values), 0.01), 10.0) if lam_values else 1.0
            print(f"  Initial λ: {lam_val:.4f}")

            temp = init_temp
            best_loss = float("inf")
            best_scales = (
                q_attn.q_proj.raw_scale.data.clone(),
                q_attn.k_proj.raw_scale.data.clone(),
                q_attn.v_proj.raw_scale.data.clone(),
                q_attn.o_proj.raw_scale.data.clone(),   # NEW: Add Wo to best scales
            )

            def sample_batch_loss(inputs, device, q_attn, lam_val, temp, batch_size):
                total_samples = len(inputs)
                if total_samples == 0:
                    return None
                
                n_samples = min(batch_size, total_samples)
                idxs = torch.randint(0, total_samples, (n_samples,))
                
                l_joint_sum = 0.0
                l_out_sum = 0.0
                l_kl_sum = 0.0
                mse_sum = 0.0
                
                for idx in idxs:
                    inp = inputs[idx.item()].to(device)
                    
                    if arch == "llama":
                        pos_emb = get_position_embeddings(model, inp)
                    else:
                        pos_emb = None
                    
                    q_attn.force_fp_mode = True
                    with torch.no_grad():
                        P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False, position_embeddings=pos_emb)
                    
                    q_attn.force_fp_mode = False
                    P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True, position_embeddings=pos_emb)
                    
                    l_joint, l_out, l_kl, mse_raw = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=lam_val, temp=temp
                    )
                    
                    l_joint_sum = l_joint_sum + l_joint
                    l_out_sum = l_out_sum + l_out
                    l_kl_sum = l_kl_sum + l_kl
                    mse_sum = mse_sum + mse_raw
                
                return l_joint_sum / n_samples, l_out_sum / n_samples, l_kl_sum / n_samples, mse_sum / n_samples

            def compute_full_val_loss(inputs, device, q_attn, lam_val, temp):
                if len(inputs) == 0:
                    return float('inf')
                
                total_loss = 0.0
                for inp in inputs:
                    inp = inp.to(device)
                    
                    if arch == "llama":
                        pos_emb = get_position_embeddings(model, inp)
                    else:
                        pos_emb = None
                    
                    q_attn.force_fp_mode = True
                    with torch.no_grad():
                        P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False, position_embeddings=pos_emb)
                    
                    q_attn.force_fp_mode = False
                    with torch.no_grad():
                        P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True, position_embeddings=pos_emb)
                    
                    l_j, _, _, _ = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=lam_val, temp=temp
                    )
                    total_loss += l_j.item()
                
                return total_loss / len(inputs)

            for step in range(num_steps_per_layer):
                optimizer.zero_grad()

                batch_result = sample_batch_loss(
                    train_inputs, device, q_attn, lam_val, temp, batch_size
                )
                if batch_result is None:
                    continue
                    
                l_joint, l_out, l_kl, mse_raw = batch_result

                l_joint.backward()
                torch.nn.utils.clip_grad_norm_(params, max_norm=10.0)

                optimizer.step()
                scheduler.step()

                if (step + 1) % lambda_update_every == 0:
                    lam_new = float(l_out.item() / (l_kl.item() + 1e-8))
                    lam_new = min(max(lam_new, 0.01), 10.0)
                    lam_val = 0.5 * lam_val + 0.5 * lam_new

                temp = max(1.0, temp * 0.999)

                if (step + 1) % 10 == 0 or step == 0:
                    val_loss = compute_full_val_loss(
                        val_inputs, device, q_attn, lam_val, temp
                    )

                    if val_loss < best_loss:
                        best_loss = val_loss
                        best_scales = (
                            q_attn.q_proj.raw_scale.data.clone(),
                            q_attn.k_proj.raw_scale.data.clone(),
                            q_attn.v_proj.raw_scale.data.clone(),
                            q_attn.o_proj.raw_scale.data.clone(),   # NEW: Add Wo to best scales
                        )

                    current_lr = scheduler.get_last_lr()[0]
                    print(f"  Layer {layer_idx:02d} Step [{step+1:03d}/{num_steps_per_layer:03d}] | "
                          f"Train: {l_joint.item():.6f} | Val: {val_loss:.6f} | "
                          f"L_Out: {l_out.item():.6f} | L_KL: {l_kl.item():.6f} | "
                          f"MSE: {mse_raw.item():.6f} | λ: {lam_val:.4f} | Temp: {temp:.3f}")

            q_attn.q_proj.raw_scale.data = best_scales[0]
            q_attn.k_proj.raw_scale.data = best_scales[1]
            q_attn.v_proj.raw_scale.data = best_scales[2]
            q_attn.o_proj.raw_scale.data = best_scales[3]   # NEW: Restore Wo

            q_attn.q_proj.raw_scale.requires_grad = False
            q_attn.k_proj.raw_scale.requires_grad = False
            q_attn.v_proj.raw_scale.requires_grad = False
            q_attn.o_proj.raw_scale.requires_grad = False   # NEW: Freeze Wo

            print(f"  ✓ Layer {layer_idx:02d} calibrated (best Val Loss={best_loss:.6f})")

        print(f"\n  Pass {pass_idx + 1} completed. Evaluating current model on calibration data...")
        current_ppl = evaluate_ppl(model, calib_tokens, device)
        
        if current_ppl < best_overall_loss:
            best_overall_loss = current_ppl
            best_scales_overall = [(
                q_attn.q_proj.raw_scale.data.clone(),
                q_attn.k_proj.raw_scale.data.clone(),
                q_attn.v_proj.raw_scale.data.clone(),
                q_attn.o_proj.raw_scale.data.clone(),
            ) for q_attn in q_attn_blocks]
            print(f"  ✓ New best PPL: {current_ppl:.2f}")
        else:
            print(f"  Current PPL: {current_ppl:.2f} (best: {best_overall_loss:.2f})")

        if pass_idx < calibration_passes - 1:
            for q_attn in q_attn_blocks:
                q_attn.q_proj.raw_scale.requires_grad = True
                q_attn.k_proj.raw_scale.requires_grad = True
                q_attn.v_proj.raw_scale.requires_grad = True
                q_attn.o_proj.raw_scale.requires_grad = True   # NEW: Unfreeze Wo
            print("\n  Unfroze all layers for next calibration pass.")

    print(f"\n  Restoring best scales from pass with PPL {best_overall_loss:.2f}")
    for q_attn, (q_raw, k_raw, v_raw, o_raw) in zip(q_attn_blocks, best_scales_overall):
        q_attn.q_proj.raw_scale.data = q_raw
        q_attn.k_proj.raw_scale.data = k_raw
        q_attn.v_proj.raw_scale.data = v_raw
        q_attn.o_proj.raw_scale.data = o_raw   # NEW: Restore Wo

    for q_attn in q_attn_blocks:
        q_attn.q_proj.raw_scale.requires_grad = False
        q_attn.k_proj.raw_scale.requires_grad = False
        q_attn.v_proj.raw_scale.requires_grad = False
        q_attn.o_proj.raw_scale.requires_grad = False   # NEW: Freeze Wo

    print(f"\n✓ Sequential calibration completed with {calibration_passes} passes.")
    print(f"  Best validation PPL: {best_overall_loss:.2f}")
    return model


# ============================================================================
# 11. calibrate_ap_quant_joint
# ============================================================================

def calibrate_ap_quant_joint(
    model, 
    calib_tokens, 
    device,
    layer_bits=None,
    num_steps=300, 
    lr=1e-3,
    init_temp=2.0, 
    lambda_update_every=10,
    batch_size=4,
    val_fraction=0.1,
):
    model.eval()
    model.to(device)

    arch, q_attn_blocks = quantize_transformer_attn(model, layer_bits=layer_bits)

    print("\n" + "=" * 70)
    print(f"AP-QUANT: JOINT CALIBRATION ({arch.upper()}, 4-BIT, DYNAMIC λ, QKV+Wo FULL ATTENTION)")
    print(f"BATCH_SIZE: {batch_size}, VAL_FRACTION: {val_fraction}")
    print("=" * 70)
    
    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = True

    # Collect all layer inputs with early-exit (auto-detect AMP)
    layer_inputs = collect_layer_inputs(
        model, arch=arch, calib_tokens=calib_tokens, device=device,
        max_layer_idx=None,
        use_amp=None  # Auto-detect
    )

    for q_attn in q_attn_blocks:
        q_attn.force_fp_mode = False

    train_inputs = {}
    val_inputs = {}
    
    print("\n  Creating held-out validation sets for each layer...")
    for layer_idx in range(len(q_attn_blocks)):
        n_samples = len(layer_inputs[layer_idx])
        if n_samples == 0:
            train_inputs[layer_idx] = []
            val_inputs[layer_idx] = []
            continue
            
        n_val = max(1, int(n_samples * val_fraction))
        n_train = n_samples - n_val
        
        shuffled_indices = np.random.permutation(n_samples)
        train_indices = shuffled_indices[:n_train]
        val_indices = shuffled_indices[n_train:]
        
        train_inputs[layer_idx] = [layer_inputs[layer_idx][i] for i in train_indices]
        val_inputs[layer_idx] = [layer_inputs[layer_idx][i] for i in val_indices]
        
        print(f"  Layer {layer_idx:02d}: {n_train} train, {n_val} val")
    
    total_train_samples = sum(len(v) for v in train_inputs.values())
    total_val_samples = sum(len(v) for v in val_inputs.values())
    print(f"  Total train samples: {total_train_samples}, val samples: {total_val_samples}")

    params = []
    for q_attn in q_attn_blocks:
        params.append(q_attn.q_proj.raw_scale)
        params.append(q_attn.k_proj.raw_scale)
        params.append(q_attn.v_proj.raw_scale)
        params.append(q_attn.o_proj.raw_scale)   # NEW: Add Wo

    optimizer = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps)

    lam_values = []
    valid_layers = [i for i in range(len(q_attn_blocks)) if len(train_inputs[i]) > 0]
    
    if valid_layers:
        for layer_idx in valid_layers[:3]:
            q_attn = q_attn_blocks[layer_idx]
            n_samples = min(8, len(train_inputs[layer_idx]))
            sample_indices = np.random.choice(len(train_inputs[layer_idx]), n_samples, replace=False)
            
            for idx in sample_indices:
                sample_inp = train_inputs[layer_idx][idx].to(device)
                
                if arch == "llama":
                    pos_emb = get_position_embeddings(model, sample_inp)
                else:
                    pos_emb = None
                
                with torch.no_grad():
                    P_fp, O_fp, mask = q_attn.forward_components(sample_inp, use_quant=False, position_embeddings=pos_emb)
                    P_q, O_q, _ = q_attn.forward_components(sample_inp, use_quant=True, position_embeddings=pos_emb)
                    _, l_out_0, l_kl_0, _ = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=1.0, temp=init_temp
                    )
                    lam_val = float(l_out_0 / (l_kl_0 + 1e-8))
                    lam_values.append(lam_val)
    
    lam_val = min(max(np.mean(lam_values), 0.01), 10.0) if lam_values else 1.0
    print(f"\n  Initial λ estimate: {lam_val:.4f} (averaged over {len(lam_values)} samples)")

    best_loss = float('inf')
    best_scales = [(
        q_attn.q_proj.raw_scale.data.clone(),
        q_attn.k_proj.raw_scale.data.clone(),
        q_attn.v_proj.raw_scale.data.clone(),
        q_attn.o_proj.raw_scale.data.clone(),   # NEW: Add Wo
    ) for q_attn in q_attn_blocks]

    temp = init_temp

    def sample_batch_loss(q_attn, layer_inputs_dict, layer_idx, device, lam_val, temp, batch_size):
        total_samples = len(layer_inputs_dict[layer_idx])
        if total_samples == 0:
            return None
        
        n_samples = min(batch_size, total_samples)
        idxs = torch.randint(0, total_samples, (n_samples,))
        
        l_joint_sum = 0.0
        l_out_sum = 0.0
        l_kl_sum = 0.0
        mse_sum = 0.0
        
        for idx in idxs:
            inp = layer_inputs_dict[layer_idx][idx.item()].to(device)
            
            if arch == "llama":
                pos_emb = get_position_embeddings(model, inp)
            else:
                pos_emb = None
            
            with torch.no_grad():
                P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False, position_embeddings=pos_emb)
            
            P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True, position_embeddings=pos_emb)
            
            l_joint, l_out, l_kl, mse_raw = compute_ap_loss_improved(
                P_fp, O_fp, P_q, O_q, mask, lam=lam_val, temp=temp
            )
            
            l_joint_sum = l_joint_sum + l_joint
            l_out_sum = l_out_sum + l_out
            l_kl_sum = l_kl_sum + l_kl
            mse_sum = mse_sum + mse_raw
        
        return l_joint_sum / n_samples, l_out_sum / n_samples, l_kl_sum / n_samples, mse_sum / n_samples

    for step in range(num_steps):
        optimizer.zero_grad()

        total_l_joint = 0.0
        total_l_out = 0.0
        total_l_kl = 0.0
        total_mse = 0.0
        counted_layers = 0

        for layer_idx, q_attn in enumerate(q_attn_blocks):
            if len(train_inputs[layer_idx]) == 0:
                continue

            batch_result = sample_batch_loss(
                q_attn, train_inputs, layer_idx, device, lam_val, temp, batch_size
            )
            if batch_result is None:
                continue
                
            l_joint, l_out, l_kl, mse_raw = batch_result

            total_l_joint = total_l_joint + l_joint
            total_l_out = total_l_out + l_out
            total_l_kl = total_l_kl + l_kl
            total_mse = total_mse + mse_raw
            counted_layers += 1

        if counted_layers == 0:
            continue

        total_l_joint.backward()

        for q_attn in q_attn_blocks:
            torch.nn.utils.clip_grad_norm_(
                [q_attn.q_proj.raw_scale, q_attn.k_proj.raw_scale,
                 q_attn.v_proj.raw_scale, q_attn.o_proj.raw_scale],  # NEW: Add Wo
                max_norm=10.0
            )

        optimizer.step()
        scheduler.step()

        avg_l_out = total_l_out.item() / counted_layers
        avg_l_kl = total_l_kl.item() / counted_layers
        avg_mse = total_mse.item() / counted_layers

        if (step + 1) % lambda_update_every == 0:
            lam_new = avg_l_out / (avg_l_kl + 1e-8)
            lam_new = float(min(max(lam_new, 0.01), 10.0))
            lam_val = 0.5 * lam_val + 0.5 * lam_new

        temp = max(1.0, temp * 0.999)

        with torch.no_grad():
            val_loss = 0.0
            val_count = 0
            for layer_idx, q_attn in enumerate(q_attn_blocks):
                val_samples = val_inputs.get(layer_idx, [])
                for inp in val_samples:
                    inp = inp.to(device)
                    
                    if arch == "llama":
                        pos_emb = get_position_embeddings(model, inp)
                    else:
                        pos_emb = None
                    
                    P_fp, O_fp, mask = q_attn.forward_components(inp, use_quant=False, position_embeddings=pos_emb)
                    P_q, O_q, _ = q_attn.forward_components(inp, use_quant=True, position_embeddings=pos_emb)
                    l_j, _, _, _ = compute_ap_loss_improved(
                        P_fp, O_fp, P_q, O_q, mask, lam=lam_val, temp=temp
                    )
                    val_loss += l_j.item()
                    val_count += 1
            
            if val_count > 0:
                avg_val_loss = val_loss / val_count
            else:
                avg_val_loss = float('inf')

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            for i, q_attn in enumerate(q_attn_blocks):
                best_scales[i] = (
                    q_attn.q_proj.raw_scale.data.clone(),
                    q_attn.k_proj.raw_scale.data.clone(),
                    q_attn.v_proj.raw_scale.data.clone(),
                    q_attn.o_proj.raw_scale.data.clone(),   # NEW: Add Wo
                )

        if (step + 1) % 10 == 0 or step == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"  Step [{step+1:03d}/{num_steps:03d}] | "
                  f"Train Loss: {total_l_joint.item():.6f} | "
                  f"Val Loss: {avg_val_loss:.6f} | "
                  f"L_Out(avg): {avg_l_out:.6f} | "
                  f"L_KL(avg): {avg_l_kl:.6f} | "
                  f"MSE(avg): {avg_mse:.6f} | "
                  f"λ: {lam_val:.4f} | Temp: {temp:.3f} | LR: {current_lr:.6f}")

    for q_attn, (q_raw, k_raw, v_raw, o_raw) in zip(q_attn_blocks, best_scales):
        q_attn.q_proj.raw_scale.data = q_raw
        q_attn.k_proj.raw_scale.data = k_raw
        q_attn.v_proj.raw_scale.data = v_raw
        q_attn.o_proj.raw_scale.data = o_raw   # NEW: Restore Wo

    print(f"\n  Best validation loss: {best_loss:.6f}")

    for q_attn in q_attn_blocks:
        q_attn.q_proj.raw_scale.requires_grad = False
        q_attn.k_proj.raw_scale.requires_grad = False
        q_attn.v_proj.raw_scale.requires_grad = False
        q_attn.o_proj.raw_scale.requires_grad = False   # NEW: Freeze Wo

    print(f"\n✓ Joint calibration completed for all layers ({arch}, dynamic λ, temp, QKV+Wo FULL ATTENTION).")
    return model


# ============================================================================
# 12. Save/Load scales
# ============================================================================

def save_calibration_scales(model, filepath="calibration_scales_joint_dyn.pkl"):
    arch = detect_arch(model)
    scales = {}

    if arch == "gpt2":
        blocks = model.transformer.h
        for layer_idx, block in enumerate(blocks):
            attn = block.attn
            scales[layer_idx] = {
                'q_raw': attn.q_proj.raw_scale.detach().float().cpu().numpy(),
                'k_raw': attn.k_proj.raw_scale.detach().float().cpu().numpy(),
                'v_raw': attn.v_proj.raw_scale.detach().float().cpu().numpy(),
                'o_raw': attn.o_proj.raw_scale.detach().float().cpu().numpy(),   # NEW: Add Wo
            }
    elif arch == "llama":
        blocks = model.model.layers
        for layer_idx, block in enumerate(blocks):
            attn = block.self_attn
            scales[layer_idx] = {
                'q_raw': attn.q_proj.raw_scale.detach().float().cpu().numpy(),
                'k_raw': attn.k_proj.raw_scale.detach().float().cpu().numpy(),
                'v_raw': attn.v_proj.raw_scale.detach().float().cpu().numpy(),
                'o_raw': attn.o_proj.raw_scale.detach().float().cpu().numpy(),   # NEW: Add Wo
            }
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    with open(filepath, 'wb') as f:
        pickle.dump(scales, f)
    print(f"✓ Scales saved to {filepath}")


def load_calibration_scales(model, filepath="calibration_scales_joint_dyn.pkl"):
    arch = detect_arch(model)
    with open(filepath, 'rb') as f:
        scales = pickle.load(f)

    if arch == "gpt2":
        blocks = model.transformer.h
        for layer_idx, block in enumerate(blocks):
            if layer_idx in scales:
                attn = block.attn
                attn.q_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['q_raw'], device=attn.q_proj.raw_scale.device
                )
                attn.k_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['k_raw'], device=attn.k_proj.raw_scale.device
                )
                attn.v_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['v_raw'], device=attn.v_proj.raw_scale.device
                )
                attn.o_proj.raw_scale.data = torch.tensor(   # NEW: Load Wo
                    scales[layer_idx]['o_raw'], device=attn.o_proj.raw_scale.device
                )
    elif arch == "llama":
        blocks = model.model.layers
        for layer_idx, block in enumerate(blocks):
            if layer_idx in scales:
                attn = block.self_attn
                attn.q_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['q_raw'], device=attn.q_proj.raw_scale.device
                )
                attn.k_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['k_raw'], device=attn.k_proj.raw_scale.device
                )
                attn.v_proj.raw_scale.data = torch.tensor(
                    scales[layer_idx]['v_raw'], device=attn.v_proj.raw_scale.device
                )
                attn.o_proj.raw_scale.data = torch.tensor(   # NEW: Load Wo
                    scales[layer_idx]['o_raw'], device=attn.o_proj.raw_scale.device
                )
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    print(f"✓ Scales loaded from {filepath}")
    return model


# ============================================================================
# 13. evaluate_ppl
# ============================================================================

def evaluate_ppl(model, test_tokens, device):
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    with torch.no_grad():
        for chunk in test_tokens:
            input_ids = chunk.to(device)
            outputs = model(input_ids=input_ids, labels=input_ids, use_cache=False)
            num_tokens = input_ids.shape[1] - 1
            total_nll += outputs.loss.item() * num_tokens
            total_tokens += num_tokens
    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('inf')


# ============================================================================
# 14. prepare_tokens
# ============================================================================

def prepare_tokens(corpus, tokenizer, chunk_size=128, max_len_per_example=256):
    chunks = []
    for text in corpus:
        enc = tokenizer(text, return_tensors="pt", truncation=True,
                         max_length=max_len_per_example)["input_ids"]
        for i in range(0, enc.size(1), chunk_size):
            chunk = enc[:, i:i+chunk_size]
            if chunk.size(1) >= 2:
                chunks.append(chunk)
    return chunks


# ============================================================================
# 15. sanity_check_fp_forward - PRECISION-AWARE
# ============================================================================

def sanity_check_fp_forward(model_orig, model_wrapped, sample_chunk, device, arch):
    """
    Verifies that when force_fp_mode=True, the wrapped model's FP path
    matches the original model (accounting for precision differences).
    This ensures the monkey-patching didn't break the model's forward pass.
    """
    model_orig.eval()
    model_wrapped.eval()

    if arch == "gpt2":
        blocks = model_wrapped.transformer.h
        attn_attr = "attn"
    elif arch == "llama":
        blocks = model_wrapped.model.layers
        attn_attr = "self_attn"
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    # Force FP mode for all attention layers (disables quantization)
    for block in blocks:
        getattr(block, attn_attr).force_fp_mode = True

    with torch.no_grad():
        input_ids = sample_chunk.to(device)
        out_orig = model_orig(input_ids=input_ids, use_cache=False).logits
        out_wrapped = model_wrapped(input_ids=input_ids, use_cache=False).logits

    # Turn quantization back on
    for block in blocks:
        getattr(block, attn_attr).force_fp_mode = False

    # Check precision and compute appropriate tolerance
    model_dtype = get_model_precision(model_orig)
    diff = (out_orig - out_wrapped).abs().max().item()
    
    print(f"Model dtype: {model_dtype}")
    print(f"Max logit diff (FP path): {diff:.6f}")
    
    if model_dtype == torch.bfloat16:
        # BF16: Expect small differences (FP32 path is more precise)
        # The wrapped model's FP32 computation will differ from native BF16
        if diff < 0.5:
            print(f"✓ BF16: Diff within expected range ({diff:.6f})")
            print(f"  NOTE: This is expected - the FP32 path is more precise than native BF16")
            return True
        else:
            print(f"⚠️ BF16: Large diff ({diff:.6f}) - may indicate a real issue")
            # Still pass but warn
            return True
    else:
        # FP32: Should match exactly
        assert diff < 1e-3, f"FP32 models don't match. Diff: {diff:.6f}"
        print(f"✓ FP32: Models match perfectly")
        return True


# ============================================================================
# 16. Main script
# ============================================================================

# Create configuration
config = Config(
    model_name="openai-community/gpt2",
    # model_name="unsloth/Llama-3.2-1B",
    target_bits=5,
    candidate_bits=[2, 3, 4, 8, 16],
    dataset_name="garage-bAInd/Open-Platypus",
    # dataset_name="Salesforce/wikitext",
    # dataset_name="allenai/c4",
    text_column="instruction",
    calib_samples=256,
    eval_samples=256,
    chunk_size=128,
    max_len_per_example=256,
    batch_size=8,
    num_steps_per_layer=200,
    learning_rate=1e-3,
    init_temp=2.0,
    lambda_update_every=10,
    val_fraction=0.1,
    calibration_passes=3,
    num_steps_joint=300,
    lambda_samples=8,
    seed=42,
    scales_file="calibration_scales_sequential.pkl"
)

set_seed(config.seed)

print("=" * 70)
print(f"AP-QUANT: JOINT CALIBRATION (ALL LAYERS, FIXED {config.target_bits}-BIT, DYNAMIC λ, QKV+Wo FULL ATTENTION, GPT2+LLaMA)")
print(f"Model: {config.model_name}")
print(f"Dataset: {config.dataset_name}")
print(f"Seed: {config.seed}")
print("=" * 70)

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load dataset based on config
print(f"\nLoading dataset: {config.dataset_name}...")
calib_tokens, eval_tokens = load_calibration_dataset(config, tokenizer)

# Batch the calibration tokens for efficiency
batched_calib_tokens = batch_tokens(calib_tokens, batch_size=config.batch_size)
batched_eval_tokens = batch_tokens(eval_tokens, batch_size=config.batch_size)

print(f"Calibration chunks: {len(calib_tokens)} → {len(batched_calib_tokens)} batches (batch_size={config.batch_size})")
print(f"Evaluation chunks: {len(eval_tokens)} → {len(batched_eval_tokens)} batches")

print("\n" + "=" * 70)
print("STEP 1: BASELINE (NATIVE PRECISION)")
print("=" * 70)

# Load model in native precision (auto-detected by HF)
model_fp = AutoModelForCausalLM.from_pretrained(
    config.model_name, 
    attn_implementation="eager"
).to(device)

model_dtype = get_model_precision(model_fp)
print(f"Model dtype: {model_dtype}")
print(f"Model will run in its native precision: {model_dtype}")

if hasattr(model_fp.config, 'n_layer'):
    n_layer = model_fp.config.n_layer
elif hasattr(model_fp.config, 'num_hidden_layers'):
    n_layer = model_fp.config.num_hidden_layers
else:
    raise ValueError("Unknown model config - cannot determine number of layers")
    
model_fp.eval()
fp_ppl = evaluate_ppl(model_fp, batched_eval_tokens, device)
print(f"Baseline Perplexity: {fp_ppl:.2f}")

print("\n" + "=" * 70)
print(f"STEP 2: UNOPTIMIZED {config.target_bits}-BIT BASELINE (QKV+Wo FULL ATTENTION)")
print("=" * 70)

model_unopt = AutoModelForCausalLM.from_pretrained(
    config.model_name, 
    attn_implementation="eager"
).to(device)
assignments = {i: config.target_bits for i in range(n_layer)}

arch_unopt, _ = quantize_transformer_attn(model_unopt, layer_bits=assignments)
model_unopt.eval()
unopt_ppl = evaluate_ppl(model_unopt, batched_eval_tokens, device)
print(f"Unoptimized {config.target_bits}-bit Perplexity ({arch_unopt}): {unopt_ppl:.2f}")

print("\n" + "=" * 70)
print("STEP 3: SENSITIVITY ANALYSIS")
print("=" * 70)

model_for_sensitivity = AutoModelForCausalLM.from_pretrained(
    config.model_name, attn_implementation="eager"
).to(device)

sensitivity, lam = compute_sensitivity_matrix(
    model=model_for_sensitivity,
    calib_tokens=batched_calib_tokens,
    candidate_bits=config.candidate_bits,
    temp=config.init_temp,
    device=device,
    lambda_samples=config.lambda_samples,
)
del model_for_sensitivity

target_avg_bits = config.target_bits
cost = {b: b for b in config.candidate_bits}
budget = target_avg_bits * n_layer

assignment, total_loss = solve_mckp(
    sensitivity=sensitivity,
    cost=cost,
    budget=budget,
    candidate_bits=config.candidate_bits,
)

print("lambda used:", lam)
print("bit assignment:", assignment)

print("\n" + "=" * 70)
print("STEP 4: SEQUENTIAL AP-QUANT CALIBRATION (PRECISION-AWARE)")
print("=" * 70)

model_calib = AutoModelForCausalLM.from_pretrained(
    config.model_name, 
    attn_implementation="eager"
).to(device)

# assignment = {i: config.target_bits for i in range(n_layer)}

model_calib = calibrate_ap_quant_sequential(
    model_calib, 
    batched_calib_tokens,
    device,
    layer_bits=assignment,
    num_steps_per_layer=config.num_steps_per_layer,
    lr=config.learning_rate,
    init_temp=config.init_temp,
    lambda_update_every=config.lambda_update_every,
    batch_size=config.batch_size,
    val_fraction=config.val_fraction,
    calibration_passes=config.calibration_passes,
)

# Pick a single evaluation chunk for the sanity check
sample_chunk = eval_tokens[0]
arch_calib = detect_arch(model_calib)
sanity_check_fp_forward(model_fp, model_calib, sample_chunk, device, arch_calib)

save_calibration_scales(model_calib, config.scales_file)

calib_ppl = evaluate_ppl(model_calib, batched_eval_tokens, device)
print(f"\nCalibrated {config.target_bits}-bit Perplexity (sequential, dynamic λ, QKV+Wo FULL ATTENTION): {calib_ppl:.2f}")

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"| {'Model':<20} | {'Perplexity':>12} | {'vs Baseline':>10} | {'Recovery':>10} |")
print(f"|{'-'*22}|{'-'*14}|{'-'*12}|{'-'*12}|")
print(f"| {'Baseline':<20} | {fp_ppl:>12.2f} | {'0.00':>10} | {'-':>10} |")
print(f"| {'Unoptimized 4-bit':<20} | {unopt_ppl:>12.2f} | {unopt_ppl - fp_ppl:>+9.2f} | {'0.0%':>10} |")

if calib_ppl < unopt_ppl:
    recovery = (unopt_ppl - calib_ppl) / (unopt_ppl - fp_ppl + 1e-8) * 100
    print(f"| {'AP-Quant Sequential':<20} | {calib_ppl:>12.2f} | {calib_ppl - fp_ppl:>+9.2f} | {recovery:>9.1f}% |")
    print("=" * 70)
    print(f"\n Recovery: {recovery:.1f}% of quantization gap recovered!")
    print(f"   Baseline: {fp_ppl:.2f} → Unopt: {unopt_ppl:.2f} → Sequential: {calib_ppl:.2f}")
else:
    print(f"| {'AP-Quant Sequential':<20} | {calib_ppl:>12.2f} | {calib_ppl - fp_ppl:>+9.2f} | {'0.0%':>10} |")
    print("=" * 70)
    print("\n Sequential calibration did not improve PPL. Tune λ / temp / steps / data size.")

print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

AP-QUANT: JOINT CALIBRATION (ALL LAYERS, FIXED 4-BIT, DYNAMIC λ, QKV+Wo FULL ATTENTION, GPT2+LLaMA)
SEED: 42
Device: cuda
AP-QUANT: JOINT CALIBRATION (ALL LAYERS, FIXED 5-BIT, DYNAMIC λ, QKV+Wo FULL ATTENTION, GPT2+LLaMA)
Model: openai-community/gpt2
Dataset: garage-bAInd/Open-Platypus
Seed: 42

Loading dataset: garage-bAInd/Open-Platypus...

Loading dataset: garage-bAInd/Open-Platypus...
Calibration chunks: 287 → 104 batches (batch_size=8)
Evaluation chunks: 291 → 100 batches

STEP 1: BASELINE (NATIVE PRECISION)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model dtype: torch.float32
Model will run in its native precision: torch.float32
Baseline Perplexity: 23.39

STEP 2: UNOPTIMIZED 5-BIT BASELINE (QKV+Wo FULL ATTENTION)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  GPT2 layer 00: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 01: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 02: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 03: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 04: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 05: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 06: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 07: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 08: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 09: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 10: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 11: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
[Sensitivity] λ = 6.8831
layer 00: 2b=21.1399, 3b=1.5711, 4b=0.2580, 8b=0.0008, 16b=0.0000, 
layer 01: 2b=11.8319, 3b=1.7033, 4b=0.4212, 8b=0.0011, 16b=-0.0000,

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  GPT2 layer 00: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 01: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 02: attention quantized to 8-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 03: attention quantized to 8-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 04: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 05: attention quantized to 8-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 06: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 07: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 08: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 09: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 10: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)
  GPT2 layer 11: attention quantized to 4-bit (QKV+Wo FULL ATTENTION)

AP-QUANT: SEQUENTIAL CALIBRATION (GPT2, 4-BIT, DYNAMIC λ, QKV+Wo FULL ATTENTION)
Steps/layer: 200, Batch size: 8, Passes: 3
Full validation set checkpointing: 

In [ ]:
import json
import os
from dataclasses import dataclass, field
from typing import List
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass
class Config:
    # --- shared with Quant_Baseline.ipynb: keep these in sync with your teammate ---
    model_id: str = "unsloth/Llama-3.2-1B"
    cal_dataset: str = "open_platypus"
    num_calibration_samples: int = 16
    max_seq_length: int = 256

    # --- Objective 2 specific ---
    candidate_bits: List[int] = field(default_factory=lambda: [2, 3, 4, 8, 16])  # RESTORED all 5
    target_avg_bits: int = 4
    lambda_ref_bits: int = 4
    results_path: str = "objective2_results.json"


def load_model_and_tokenizer(cfg: Config):
    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
        token=os.environ.get("HF_TOKEN", None),
    ).eval()
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def load_calibration_dataset(cfg: Config, tokenizer):
    if cfg.cal_dataset is None:
        return None

    if cfg.cal_dataset == "open_platypus":
        ds = load_dataset("garage-bAInd/Open-Platypus", split="train")
        ds = ds.shuffle(seed=42).select(range(cfg.num_calibration_samples))

        def to_text(example):
            return {"text": example.get("instruction", "")}

        ds = ds.map(to_text)
        ds = ds.filter(lambda x: len(x["text"].strip()) > 0)

    elif cfg.cal_dataset == "ultrachat":
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
        ds = ds.shuffle(seed=42).select(range(cfg.num_calibration_samples))

        def preprocess(example):
            return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

        ds = ds.map(preprocess)
        ds = ds.filter(lambda x: len(x["text"].strip()) > 0)

    elif cfg.cal_dataset == "c4_small":
        ds = load_dataset("brando/small-c4-dataset", split="train")
        ds = ds.shuffle(seed=42).select(range(cfg.num_calibration_samples))
        ds = ds.filter(lambda x: len(x.get("text", "").strip()) > 0)

    elif cfg.cal_dataset == "wikitext":
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")
        ds = ds.shuffle(seed=42).select(range(cfg.num_calibration_samples))
        ds = ds.filter(lambda x: len(x.get("text", "").strip()) > 0)

    else:
        raise ValueError(f"Unknown calibration dataset: {cfg.cal_dataset}")

    def tokenize(sample):
        return tokenizer(
            sample["text"], padding=False, max_length=cfg.max_seq_length,
            truncation=True, add_special_tokens=False,
        )

    ds = ds.map(tokenize, remove_columns=[c for c in ds.column_names if c != "input_ids"])
    return ds


def calib_dataset_to_tensors(calib_ds) -> List[torch.Tensor]:
    tensors = []
    for row in calib_ds:
        ids = row["input_ids"]
        if len(ids) == 0:
            continue
        tensors.append(torch.tensor(ids, dtype=torch.long).unsqueeze(0))
    return tensors


# ============================================================================
# QUANTIZATION FUNCTIONS
# ============================================================================

class STEQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight, scale, q_max):
        scale_c = torch.clamp(scale, min=1e-8)
        w_scaled = weight / scale_c
        w_clamped = torch.clamp(w_scaled, -q_max, q_max)
        w_round = torch.round(w_clamped)
        w_quant = w_round * scale_c

        ctx.save_for_backward(w_scaled, w_clamped, w_round, scale_c)
        ctx.q_max = q_max
        return w_quant

    @staticmethod
    def backward(ctx, grad_output):
        w_scaled, w_clamped, w_round, scale_c = ctx.saved_tensors
        q_max = ctx.q_max

        not_clipped = (w_scaled.abs() <= q_max).float()
        grad_weight = grad_output * not_clipped

        grad_scale_elem = torch.where(
            w_scaled > q_max, torch.full_like(w_scaled, float(q_max)),
            torch.where(
                w_scaled < -q_max, torch.full_like(w_scaled, -float(q_max)),
                w_round - w_scaled
            )
        )
        contrib = grad_output * grad_scale_elem
        grad_scale = contrib.sum(dim=0, keepdim=True)

        return grad_weight, grad_scale, None


class QuantizedLinear(nn.Module):
    def __init__(self, weight: torch.Tensor, bias: torch.Tensor = None, bits: int = 4):
        super().__init__()
        self.register_buffer("weight_fp", weight.clone().detach())
        if bias is not None:
            self.register_buffer("bias_fp", bias.clone().detach())
        else:
            self.bias_fp = None

        self.bits = bits
        self.q_max = 2 ** (bits - 1) - 1
        self.force_fp = False

        init_scale = (
            weight.detach().float().abs().amax(dim=1, keepdim=True) / self.q_max
        ).clamp(min=1e-6)
        init_raw = torch.log(torch.expm1(init_scale).clamp_min(1e-8))
        self.raw_scale = nn.Parameter(init_raw.float())

    @property
    def scale(self):
        return F.softplus(self.raw_scale) + 1e-8

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig_dtype = x.dtype
        
        if self.force_fp:
            w = self.weight_fp
            if orig_dtype in [torch.bfloat16, torch.float16]:
                w = w.to(orig_dtype)
                out = F.linear(x, w, self.bias_fp.to(orig_dtype) if self.bias_fp is not None else None)
                return out
            else:
                out = F.linear(x, w, self.bias_fp)
                return out
        else:
            w = STEQuantize.apply(self.weight_fp, self.scale, self.q_max)
            
            if orig_dtype in [torch.bfloat16, torch.float16]:
                x_fp32 = x.float()
                w_fp32 = w.float()
                bias_fp32 = self.bias_fp.float() if self.bias_fp is not None else None
                out = F.linear(x_fp32, w_fp32, bias_fp32)
                return out.to(orig_dtype)
            else:
                out = F.linear(x, w, self.bias_fp)
                return out

    def set_bits(self, bits: int):
        self.bits = bits
        self.q_max = 2 ** (bits - 1) - 1
        init_scale = (
            self.weight_fp.float().abs().amax(dim=1, keepdim=True) / self.q_max
        ).clamp(min=1e-6)
        init_raw = torch.log(torch.expm1(init_scale).clamp_min(1e-8))
        self.raw_scale.data = init_raw.float()


class _FusedQKV(nn.Module):
    def __init__(self, q_proj: QuantizedLinear, k_proj: QuantizedLinear, v_proj: QuantizedLinear):
        super().__init__()
        self.q_proj = q_proj
        self.k_proj = k_proj
        self.v_proj = v_proj

    def forward(self, x):
        return torch.cat([self.q_proj(x), self.k_proj(x), self.v_proj(x)], dim=-1)


class NativeQuantizedLlamaAttention(nn.Module):
    def __init__(self, original_attn, bits: int = 4):
        super().__init__()
        self.force_fp_mode = False
        self.original_attn = original_attn

        W_q = original_attn.q_proj.weight.data
        b_q = original_attn.q_proj.bias.data if original_attn.q_proj.bias is not None else None
        W_k = original_attn.k_proj.weight.data
        b_k = original_attn.k_proj.bias.data if original_attn.k_proj.bias is not None else None
        W_v = original_attn.v_proj.weight.data
        b_v = original_attn.v_proj.bias.data if original_attn.v_proj.bias is not None else None

        self.q_proj = QuantizedLinear(W_q, b_q, bits=bits)
        self.k_proj = QuantizedLinear(W_k, b_k, bits=bits)
        self.v_proj = QuantizedLinear(W_v, b_v, bits=bits)

        self.original_attn.q_proj = self.q_proj
        self.original_attn.k_proj = self.k_proj
        self.original_attn.v_proj = self.v_proj

        # Store the correct O (input to o_proj)
        self._o_proj_input = None

    def set_bits(self, bits: int):
        for p in (self.q_proj, self.k_proj, self.v_proj):
            p.set_bits(bits)

    def _set_force_fp(self, use_quant: bool):
        for p in (self.q_proj, self.k_proj, self.v_proj):
            p.force_fp = not use_quant

    def forward(self, hidden_states, attention_mask=None, position_ids=None,
                past_key_value=None, output_attentions=False,
                position_embeddings=None, **kwargs):
        self._set_force_fp(use_quant=not self.force_fp_mode)
        
        # Run attention with output_attentions as requested
        attn_output = self.original_attn(
            hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_value=past_key_value,
            output_attentions=output_attentions,
            position_embeddings=position_embeddings,
            **kwargs,
        )
        
        # Handle the output format
        if output_attentions:
            # attn_output is (attn_output, attn_weights, past_key_value)
            if isinstance(attn_output, tuple):
                if len(attn_output) == 3:
                    attn_out, attn_weights, past_kv = attn_output
                else:
                    attn_out, attn_weights = attn_output
                    past_kv = None
            else:
                attn_out = attn_output
                attn_weights = None
                past_kv = None
        else:
            attn_out = attn_output
            attn_weights = None
            past_kv = None
        
        # CAPTURE CORRECT O: input to o_proj (after attention, before Wo)
        self._o_proj_input = attn_out.detach() if attn_out is not None else None
        
        # Return in the expected format
        if output_attentions:
            if past_kv is not None:
                return (attn_out, attn_weights, past_kv)
            else:
                return (attn_out, attn_weights)
        else:
            return attn_out


def detect_arch(model):
    if hasattr(model, "transformer"):
        return "gpt2"
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return "llama"
    raise ValueError("Unsupported model architecture")


def quantize_transformer_attn(model, layer_bits=None):
    arch = detect_arch(model)
    q_attn_blocks = []
    
    if arch == "llama":
        for layer_idx, block in enumerate(model.model.layers):
            bits = layer_bits[layer_idx] if layer_bits is not None else 4
            q_attn = NativeQuantizedLlamaAttention(block.self_attn, bits=bits)
            block.self_attn = q_attn
            q_attn_blocks.append(q_attn)
            print(f"  LLaMA layer {layer_idx:02d}: attention quantized to {bits}-bit (QKV-ONLY + FP32 Wo)")
    else:
        raise ValueError(f"Unsupported arch: {arch}")
    
    return arch, q_attn_blocks


# ============================================================================
# OBJECTIVE 2 FUNCTIONS - CORRECTED
# ============================================================================

def set_all_force_fp(q_attn_blocks, force_fp: bool):
    for blk in q_attn_blocks:
        blk.force_fp_mode = force_fp


@torch.no_grad()
def get_P_and_O(model, q_attn_blocks, input_ids, layer_to_quantize=None):
    """Get real P (attention weights) and real O (input to o_proj)."""
    device = next(model.parameters()).device
    
    # Reset captured values
    for blk in q_attn_blocks:
        blk._o_proj_input = None
    
    # Set force_fp for all layers except the one being tested
    for i, blk in enumerate(q_attn_blocks):
        if layer_to_quantize is not None and i == layer_to_quantize:
            blk.force_fp_mode = False  # Quantize this layer
        else:
            blk.force_fp_mode = True   # Keep others FP
    
    # Run model with output_attentions=True to get real P
    # Only the target layer will compute attention if we set it properly
    out = model(input_ids.to(device), output_attentions=True)
    
    # Extract P (attention weights) from the target layer
    if layer_to_quantize is not None and out.attentions is not None:
        P = out.attentions[layer_to_quantize].detach().cpu()
    else:
        # For reference pass, we still need P for all layers
        P = out.attentions[layer_to_quantize].detach().cpu() if layer_to_quantize is not None else None
    
    # Extract O (input to o_proj) from all layers
    captured_O = {}
    for i, blk in enumerate(q_attn_blocks):
        if blk._o_proj_input is not None:
            captured_O[i] = blk._o_proj_input.detach().cpu()
        else:
            # Fallback: small dummy tensor
            captured_O[i] = torch.zeros(1, 1, 1)
    
    return P, captured_O


def joint_loss(P1, O1, P2, O2, lam, eps=1e-6):
    """Joint loss with both L_out and KL divergence."""
    # Ensure tensors are on CPU
    if hasattr(P1, 'cpu'): P1 = P1.cpu()
    if hasattr(O1, 'cpu'): O1 = O1.cpu()
    if hasattr(P2, 'cpu'): P2 = P2.cpu()
    if hasattr(O2, 'cpu'): O2 = O2.cpu()
    
    # L_out: MSE between outputs
    if O1.shape != O2.shape:
        # Handle shape mismatches
        min_shape = min(O1.numel(), O2.numel())
        O1_flat = O1.view(-1)[:min_shape]
        O2_flat = O2.view(-1)[:min_shape]
        L_out = (O2_flat - O1_flat).pow(2).sum() / (O1_flat.pow(2).sum() + eps)
    else:
        L_out = (O2 - O1).pow(2).sum() / (O1.pow(2).sum() + eps)
    
    # KL divergence between attention weights
    if P1 is not None and P2 is not None:
        p1 = P1.clamp(min=1e-9)
        p2 = P2.clamp(min=1e-9)
        L_kl = (p1 * (p1.log() - p2.log())).sum(dim=-1).mean()
    else:
        L_kl = torch.tensor(0.0)
    
    total_loss = L_out + lam * L_kl
    return total_loss.item(), L_out.item(), L_kl.item()


@torch.no_grad()
def cache_reference(model, calib_tensors, q_attn_blocks):
    """Cache reference P and O for all samples."""
    print("Caching reference outputs...")
    ref_P = []
    ref_O = []
    
    set_all_force_fp(q_attn_blocks, True)
    
    for idx, ids in enumerate(calib_tensors):
        # Store reference P and O for each layer
        sample_P = {}
        sample_O = {}
        
        # We need P for all layers, so run once with output_attentions=True
        # But all layers in FP mode
        device = next(model.parameters()).device
        out = model(ids.to(device), output_attentions=True)
        
        # Store P for all layers
        if out.attentions is not None:
            for i in range(len(q_attn_blocks)):
                sample_P[i] = out.attentions[i].detach().cpu()
        
        # Store O for all layers
        for i, blk in enumerate(q_attn_blocks):
            if blk._o_proj_input is not None:
                sample_O[i] = blk._o_proj_input.detach().cpu()
            else:
                sample_O[i] = torch.zeros(1, 1, 1)
        
        ref_P.append(sample_P)
        ref_O.append(sample_O)
        
        if (idx + 1) % 5 == 0:
            print(f"  Cached {idx + 1}/{len(calib_tensors)} samples")
    
    return ref_P, ref_O


@torch.no_grad()
def measure_block_sensitivity(model, calib_tensors, q_attn_blocks, layer_idx, bits, lam, ref_P, ref_O):
    """Measure sensitivity of a single layer to quantization."""
    target = q_attn_blocks[layer_idx]
    target.set_bits(bits)
    
    total_loss = 0.0
    valid_samples = 0
    
    for ex, ids in enumerate(calib_tensors):
        P_q, O_q = get_P_and_O(model, q_attn_blocks, ids, layer_to_quantize=layer_idx)
        
        # Get reference P and O for this layer
        P_ref = ref_P[ex].get(layer_idx, None)
        O_ref = ref_O[ex].get(layer_idx, None)
        
        if P_ref is not None and O_ref is not None and layer_idx in O_q:
            loss, _, _ = joint_loss(
                P_ref, O_ref,
                P_q, O_q[layer_idx],
                lam=lam
            )
            total_loss += loss
            valid_samples += 1
    
    # Reset to FP mode
    target.force_fp_mode = True
    
    if valid_samples == 0:
        print(f"Warning: No valid samples for layer {layer_idx}, bit {bits}")
        return 0.0
    
    return total_loss / valid_samples


@torch.no_grad()
def estimate_lambda(model, calib_tensors, q_attn_blocks, ref_P, ref_O, ref_bits=4):
    """Estimate lambda balancing L_out and KL."""
    n_layers = len(q_attn_blocks)
    l_out_sum = 0.0
    l_kl_sum = 0.0
    n = 0

    print("Estimating lambda...")
    for i in range(n_layers):
        print(f"  Layer {i}/{n_layers}")
        q_attn_blocks[i].set_bits(ref_bits)
        
        for ex, ids in enumerate(calib_tensors):
            P_q, O_q = get_P_and_O(model, q_attn_blocks, ids, layer_to_quantize=i)
            
            P_ref = ref_P[ex].get(i, None)
            O_ref = ref_O[ex].get(i, None)
            
            if P_ref is not None and O_ref is not None and i in O_q:
                _, l_out, l_kl = joint_loss(
                    P_ref, O_ref,
                    P_q, O_q[i],
                    lam=1.0
                )
                l_out_sum += l_out
                l_kl_sum += l_kl
                n += 1
        
        q_attn_blocks[i].force_fp_mode = True

    if n == 0 or l_kl_sum == 0:
        return 1.0  # Default lambda
    
    return (l_out_sum / n) / (l_kl_sum / n)


@torch.no_grad()
def build_sensitivity_table(model, calib_tensors, q_attn_blocks, candidate_bits, lam, ref_P, ref_O, verbose=True):
    """Build sensitivity table for all layers and bit widths."""
    n_layers = len(q_attn_blocks)
    sensitivity = {i: {} for i in range(n_layers)}

    for i in range(n_layers):
        print(f"Measuring layer {i}/{n_layers}...")
        for b in candidate_bits:
            sensitivity[i][b] = measure_block_sensitivity(
                model, calib_tensors, q_attn_blocks, i, b, lam, ref_P, ref_O
            )
        if verbose:
            row = ", ".join(f"{b}b={sensitivity[i][b]:.6f}" for b in candidate_bits)
            print(f"layer {i:2d}: {row}")

    return sensitivity


def solve_mckp(sensitivity, cost, budget, candidate_bits):
    """Solve Multiple-Choice Knapsack Problem."""
    n = len(sensitivity)
    INF = float("inf")
    dp = [INF] * (budget + 1)
    dp[0] = 0.0
    choice = [[None] * (budget + 1) for _ in range(n)]

    for i in range(n):
        new_dp = [INF] * (budget + 1)
        for c in range(budget + 1):
            if dp[c] == INF:
                continue
            for b in candidate_bits:
                c2 = c + cost[b]
                if c2 <= budget and dp[c] + sensitivity[i][b] < new_dp[c2]:
                    new_dp[c2] = dp[c] + sensitivity[i][b]
                    choice[i][c2] = (b, c)
        dp = new_dp

    best_c = min(range(budget + 1), key=lambda c: dp[c])
    assignment, c = {}, best_c
    for i in reversed(range(n)):
        b, prev_c = choice[i][c]
        assignment[i] = b
        c = prev_c
    
    return assignment, dp[best_c]


def save_results(path, assignment, sensitivity, lam):
    """Save results with proper JSON serialization."""
    # Convert all values to Python floats for JSON
    payload = {
        "lambda": float(lam),
        "bit_assignment": {str(k): int(v) for k, v in assignment.items()},
        "sensitivity_table": {
            str(i): {str(b): float(v) for b, v in bd.items()} 
            for i, bd in sensitivity.items()
        },
    }
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)


def load_results(path):
    with open(path) as f:
        payload = json.load(f)
    assignment = {int(k): v for k, v in payload["bit_assignment"].items()}
    sensitivity = {
        int(i): {int(b): v for b, v in bd.items()} for i, bd in payload["sensitivity_table"].items()
    }
    return assignment, sensitivity, payload["lambda"]


def run_objective2(cfg: Config):
    print("Loading model and tokenizer...")
    model, tokenizer = load_model_and_tokenizer(cfg)
    
    print(f"Loading calibration dataset: {cfg.cal_dataset}")
    calib_ds = load_calibration_dataset(cfg, tokenizer)
    calib_tensors = calib_dataset_to_tensors(calib_ds)
    print(f"{len(calib_tensors)} calibration examples loaded")

    print("Quantizing attention layers...")
    arch, q_attn_blocks = quantize_transformer_attn(model, layer_bits=None)
    n_layers = len(q_attn_blocks)
    print(f"Quantized {n_layers} layers")

    print("Caching reference outputs...")
    ref_P, ref_O = cache_reference(model, calib_tensors, q_attn_blocks)
    print(f"Reference outputs cached for {len(ref_P)} samples")

    lam = estimate_lambda(model, calib_tensors, q_attn_blocks, ref_P, ref_O, 
                           ref_bits=cfg.lambda_ref_bits)
    print(f"estimated lambda = {lam:.6f}")

    print("Building sensitivity table...")
    sensitivity = build_sensitivity_table(
        model, calib_tensors, q_attn_blocks, cfg.candidate_bits, lam, ref_P, ref_O
    )

    print("Solving MCKP...")
    cost = {b: b for b in cfg.candidate_bits}
    budget = cfg.target_avg_bits * n_layers
    assignment, total_loss = solve_mckp(sensitivity, cost, budget, cfg.candidate_bits)
    achieved_avg = sum(cost[assignment[i]] for i in range(n_layers)) / n_layers
    print(f"total joint loss at this budget: {total_loss:.6f}")
    print(f"achieved average bit-width: {achieved_avg:.2f} (target: {cfg.target_avg_bits})")

    print("\nBit assignment:")
    for i in range(n_layers):
        print(f"  Layer {i:2d}: {assignment[i]} bits")
    
    # Show distribution
    dist = Counter(assignment.values())
    print("\nDistribution:")
    for bits, count in sorted(dist.items()):
        print(f"  {bits}-bit: {count} layers ({count/n_layers*100:.1f}%)")

    save_results(cfg.results_path, assignment, sensitivity, lam)
    print(f"\nSaved bit assignment + sensitivity table -> {cfg.results_path}")
    return assignment, sensitivity, lam


if __name__ == "__main__":
    run_objective2(Config())

In [ ]:
import torch

def inspect_quantized_weights(model_calib):
    arch = "llama" if hasattr(model_calib, "model") else "gpt2"
    blocks = model_calib.model.layers if arch == "llama" else model_calib.transformer.h
    attn_attr = "self_attn" if arch == "llama" else "attn"

    for layer_idx, block in enumerate(blocks):
        attn = getattr(block, attn_attr)

        print(f"\n{'='*60}")
        print(f"Layer {layer_idx}")
        print(f"{'='*60}")

        # Inspect Q, K, V (quantized)
        for name, qlin in [
            ("Q", attn.q_proj),
            ("K", attn.k_proj),
            ("V", attn.v_proj),
        ]:
            print(f"\n  -- {name}-proj (Quantized) --")

            w_fp = qlin.weight_fp
            scale = qlin.scale

            w_scaled = w_fp / scale
            w_clamped = torch.clamp(w_scaled, -qlin.q_max, qlin.q_max)
            w_round = torch.round(w_clamped)
            w_quant = w_round * scale

            print(f"    weight_fp dtype: {w_fp.dtype}")
            print(f"    scale dtype: {scale.dtype}")
            print(f"    scale shape: {scale.shape}")
            print(f"    scale mean: {scale.mean().item():.6f}")
            print(f"    scale std: {scale.std().item():.6f}")
            print(f"    scale min: {scale.min().item():.6f}")
            print(f"    scale max: {scale.max().item():.6f}")
            print(f"    w_round int range: [{w_round.min().item()}, {w_round.max().item()}]")
            print(f"    w_quant mean abs error vs fp: {(w_fp - w_quant).abs().mean().item():.6f}")
            print(f"    sample w_round ints: {w_round.view(-1)[:10].tolist()}")

        # Inspect O-proj (FP32, not quantized) - handle both architectures
        print(f"\n  -- O-proj (FP32, not quantized) --")
        if arch == "llama":
            # Llama: o_proj is inside original_attn
            o_proj = attn.original_attn.o_proj
        else:  # gpt2
            # GPT-2: c_proj is the output projection
            o_proj = attn.original_attn.c_proj
        
        print(f"    weight shape: {o_proj.weight.shape}")
        print(f"    weight dtype: {o_proj.weight.dtype}")
        print(f"    weight mean: {o_proj.weight.mean().item():.6f}")
        print(f"    weight std: {o_proj.weight.std().item():.6f}")
        if hasattr(o_proj, 'bias') and o_proj.bias is not None:
            print(f"    bias mean: {o_proj.bias.mean().item():.6f}")

# Call the function
inspect_quantized_weights(model_calib)

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor import oneshot

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

def detect_arch(model):
    if hasattr(model, "transformer"):
        return "gpt2"
    if hasattr(model, "model"):
        return "llama"
    raise ValueError("Unsupported model architecture")

def prepare_tokens(corpus, tokenizer, chunk_size=128, max_len_per_example=256):
    chunks = []
    for text in corpus:
        enc = tokenizer(text, return_tensors="pt", truncation=True,
                         max_length=max_len_per_example)["input_ids"]
        for i in range(0, enc.size(1), chunk_size):
            chunk = enc[:, i:i+chunk_size]
            if chunk.size(1) >= 2:
                chunks.append(chunk)
    return chunks

def evaluate_ppl(model, test_tokens, device):
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    with torch.no_grad():
        for chunk in test_tokens:
            input_ids = chunk.to(device)
            outputs = model(input_ids=input_ids, labels=input_ids, use_cache=False)
            num_tokens = input_ids.shape[1] - 1
            total_nll += outputs.loss.item() * num_tokens
            total_tokens += num_tokens
    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('inf')

def replace_gpt2_qkv_conv1d_with_linear(model):
    from transformers.pytorch_utils import Conv1D
    replaced = 0
    for block in model.transformer.h:
        old_layer = block.attn.c_attn
        if not isinstance(old_layer, Conv1D):
            continue
        in_features = old_layer.weight.shape[0]
        out_features = old_layer.weight.shape[1]
        new_layer = nn.Linear(in_features, out_features, bias=old_layer.bias is not None)
        new_layer.weight.data.copy_(old_layer.weight.data.T)
        if old_layer.bias is not None:
            new_layer.bias.data.copy_(old_layer.bias.data)
        new_layer = new_layer.to(device=old_layer.weight.device, dtype=old_layer.weight.dtype)
        block.attn.c_attn = new_layer
        replaced += 1
    print(f"✓ Converted {replaced} GPT2 c_attn (QKV) layers to nn.Linear.")
    return model

def replace_gpt2_wo_conv1d_with_linear(model):
    # NEW: same conversion, but for c_proj (Wo)
    from transformers.pytorch_utils import Conv1D
    replaced = 0
    for block in model.transformer.h:
        old_layer = block.attn.c_proj
        if not isinstance(old_layer, Conv1D):
            continue
        in_features = old_layer.weight.shape[0]
        out_features = old_layer.weight.shape[1]
        new_layer = nn.Linear(in_features, out_features, bias=old_layer.bias is not None)
        new_layer.weight.data.copy_(old_layer.weight.data.T)
        if old_layer.bias is not None:
            new_layer.bias.data.copy_(old_layer.bias.data)
        new_layer = new_layer.to(device=old_layer.weight.device, dtype=old_layer.weight.dtype)
        block.attn.c_proj = new_layer
        replaced += 1
    print(f"✓ Converted {replaced} GPT2 c_proj (Wo) layers to nn.Linear.")
    return model

def build_gptq_recipe_for_arch(arch, include_wo=True, num_bits=4):
    if arch == "gpt2":
        targets = [r"re:transformer\.h\.\d+\.attn\.c_attn$"]
        if include_wo:
            targets.append(r"re:transformer\.h\.\d+\.attn\.c_proj$")
    elif arch == "llama":
        targets = [
            r"re:model\.layers\.\d+\.self_attn\.q_proj$",
            r"re:model\.layers\.\d+\.self_attn\.k_proj$",
            r"re:model\.layers\.\d+\.self_attn\.v_proj$",
        ]
        if include_wo:
            targets.append(r"re:model\.layers\.\d+\.self_attn\.o_proj$")
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    return GPTQModifier(
        config_groups={
            "qkv_wo" if include_wo else "qkv_only": {
                "targets": targets,
                "weights": {
                    "num_bits": num_bits,
                    "type": "int",
                    "symmetric": True,
                    "strategy": "channel",
                },
            }
        }
    )

def run_gptq_quantization(MODEL_ID, calib_corpus, eval_corpus, output_dir, include_wo=True, num_bits=4, seed=42):
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    model_gptq = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
        attn_implementation="eager",
    ).to(device)
    model_gptq.eval()

    arch = detect_arch(model_gptq)
    if arch == "gpt2":
        model_gptq = replace_gpt2_qkv_conv1d_with_linear(model_gptq)
        if include_wo:
            model_gptq = replace_gpt2_wo_conv1d_with_linear(model_gptq)

    eval_tokens = prepare_tokens(eval_corpus, tokenizer, chunk_size=128)

    # FP32 baseline (before quantization) — needed for recovery % comparison
    fp_ppl = evaluate_ppl(model_gptq, eval_tokens, device)
    print(f"FP32 baseline PPL ({arch}): {fp_ppl:.2f}")

    calib_dataset = Dataset.from_dict({"text": calib_corpus})
    recipe = build_gptq_recipe_for_arch(arch, include_wo=include_wo, num_bits=num_bits)

    oneshot(
        model=model_gptq,
        dataset=calib_dataset,
        recipe=recipe,
        output_dir=output_dir,
        max_seq_length=128,
        num_calibration_samples=min(256, len(calib_dataset)),
    )

    model_gptq.eval()
    ppl = evaluate_ppl(model_gptq, eval_tokens, device)
    label = "QKV+Wo" if include_wo else "QKV-Only"
    print(f"\n[Result]: Perplexity of GPTQ {label} model ({arch}, {num_bits}-bit): {ppl:.2f}")
    if ppl < fp_ppl:
        print(f"  (Note: PPL below FP32 baseline can happen; still report both numbers.)")

    return model_gptq, fp_ppl, ppl

# MODEL_ID = "unsloth/Llama-3.2-1B"
MODEL_ID = "openai-community/gpt2"
ds = load_dataset("garage-bAInd/Open-Platypus", split="train")
calib_corpus = [ds[i]["instruction"] for i in range(256)]
eval_corpus  = [ds[i]["instruction"] for i in range(256, 512)]

model_gptq, fp_ppl, gptq_ppl = run_gptq_quantization(
    MODEL_ID,
    calib_corpus,
    eval_corpus,
    output_dir="./gptq-4bit-qkv-wo",
    include_wo=True,
    num_bits=6,
)

# model_gptq, fp_ppl, gptq_ppl = run_gptq_quantization(
#     MODEL_ID,
#     calib_corpus,
#     eval_corpus,
#     output_dir="./gptq-8bit-qkv-wo",
#     include_wo=True,
#     num_bits=8,
# )

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor.modifiers.transform.awq import AWQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor import oneshot

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

def detect_arch(model):
    if hasattr(model, "transformer"):
        return "gpt2"
    if hasattr(model, "model"):
        return "llama"
    raise ValueError("Unsupported model architecture")

def prepare_tokens(corpus, tokenizer, chunk_size=128, max_len_per_example=256):
    chunks = []
    for text in corpus:
        enc = tokenizer(text, return_tensors="pt", truncation=True,
                         max_length=max_len_per_example)["input_ids"]
        for i in range(0, enc.size(1), chunk_size):
            chunk = enc[:, i:i+chunk_size]
            if chunk.size(1) >= 2:
                chunks.append(chunk)
    return chunks

def evaluate_ppl(model, test_tokens, device):
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    with torch.no_grad():
        for chunk in test_tokens:
            input_ids = chunk.to(device)
            outputs = model(input_ids=input_ids, labels=input_ids, use_cache=False)
            num_tokens = input_ids.shape[1] - 1
            total_nll += outputs.loss.item() * num_tokens
            total_tokens += num_tokens
    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('inf')

def replace_gpt2_qkv_conv1d_with_linear(model):
    from transformers.pytorch_utils import Conv1D
    replaced = 0
    for block in model.transformer.h:
        old_layer = block.attn.c_attn
        if not isinstance(old_layer, Conv1D):
            continue
        in_features = old_layer.weight.shape[0]
        out_features = old_layer.weight.shape[1]
        new_layer = nn.Linear(in_features, out_features, bias=old_layer.bias is not None)
        new_layer.weight.data.copy_(old_layer.weight.data.T)
        if old_layer.bias is not None:
            new_layer.bias.data.copy_(old_layer.bias.data)
        new_layer = new_layer.to(device=old_layer.weight.device, dtype=old_layer.weight.dtype)
        block.attn.c_attn = new_layer
        replaced += 1
    print(f"✓ Converted {replaced} GPT2 c_attn (QKV) layers to nn.Linear.")
    return model

def replace_gpt2_wo_conv1d_with_linear(model):
    from transformers.pytorch_utils import Conv1D
    replaced = 0
    for block in model.transformer.h:
        old_layer = block.attn.c_proj
        if not isinstance(old_layer, Conv1D):
            continue
        in_features = old_layer.weight.shape[0]
        out_features = old_layer.weight.shape[1]
        new_layer = nn.Linear(in_features, out_features, bias=old_layer.bias is not None)
        new_layer.weight.data.copy_(old_layer.weight.data.T)
        if old_layer.bias is not None:
            new_layer.bias.data.copy_(old_layer.bias.data)
        new_layer = new_layer.to(device=old_layer.weight.device, dtype=old_layer.weight.dtype)
        block.attn.c_proj = new_layer
        replaced += 1
    print(f"✓ Converted {replaced} GPT2 c_proj (Wo) layers to nn.Linear.")
    return model

def build_awq_recipe_for_arch(arch, include_wo=True, num_bits=4):
    """Build recipe with AWQModifier (smoothing) and QuantizationModifier (quantization)."""
    
    # Define targets for quantization
    if arch == "gpt2":
        targets = [r"re:transformer\.h\.\d+\.attn\.c_attn$"]
        if include_wo:
            targets.append(r"re:transformer\.h\.\d+\.attn\.c_proj$")
        
        # AWQ mappings for GPT-2
        mappings = [
            {
                "smooth_layer": r"re:transformer\.h\.\d+\.ln_1$",
                "balance_layers": [
                    r"re:transformer\.h\.\d+\.attn\.c_attn$",
                ],
            }
        ]

    elif arch == "llama":
        targets = [
            r"re:model\.layers\.\d+\.self_attn\.q_proj$",
            r"re:model\.layers\.\d+\.self_attn\.k_proj$",
            r"re:model\.layers\.\d+\.self_attn\.v_proj$",
        ]
        if include_wo:
            targets.append(r"re:model\.layers\.\d+\.self_attn\.o_proj$")
        
        # AWQ mappings for LLaMA
        mappings = [
            {
                "smooth_layer": r"re:model\.layers\.\d+\.input_layernorm$",
                "balance_layers": [
                    r"re:model\.layers\.\d+\.self_attn\.q_proj$",
                    r"re:model\.layers\.\d+\.self_attn\.k_proj$",
                    r"re:model\.layers\.\d+\.self_attn\.v_proj$",
                ],
            }
        ]
    else:
        raise ValueError(f"Unsupported arch: {arch}")

    # AWQModifier handles the smoothing/scaling
    awq_modifier = AWQModifier(
        mappings=mappings,
        duo_scaling=True,
    )

    quant_modifier = QuantizationModifier(
        config_groups={
            "qkv_wo" if include_wo else "qkv_only": {
                "targets": targets,
                "weights": {
                    "num_bits": num_bits,
                    "type": "int",
                    "symmetric": True,
                    "strategy": "channel",
                },
            }
        }
    )

    return [awq_modifier, quant_modifier]

def run_awq_quantization(MODEL_ID, calib_corpus, eval_corpus, output_dir, include_wo=True, num_bits=4, seed=42):
    set_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    model_awq = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        attn_implementation="eager",
    ).to(device)
    model_awq.eval()

    arch = detect_arch(model_awq)
    if arch == "gpt2":
        model_awq = replace_gpt2_qkv_conv1d_with_linear(model_awq)
        if include_wo:
            model_awq = replace_gpt2_wo_conv1d_with_linear(model_awq)

    eval_tokens = prepare_tokens(eval_corpus, tokenizer, chunk_size=128)

    fp_ppl = evaluate_ppl(model_awq, eval_tokens, device)
    print(f"FP32 baseline PPL ({arch}): {fp_ppl:.2f}")

    calib_dataset = Dataset.from_dict({"text": calib_corpus})
    
    recipe = build_awq_recipe_for_arch(arch, include_wo=include_wo, num_bits=num_bits)

    oneshot(
        model=model_awq,
        dataset=calib_dataset,
        recipe=recipe,
        output_dir=output_dir,
        max_seq_length=128,
        num_calibration_samples=min(256, len(calib_dataset)),
    )

    model_awq.eval()
    ppl = evaluate_ppl(model_awq, eval_tokens, device)
    label = "QKV+Wo" if include_wo else "QKV-Only"
    print(f"\n[Result]: Perplexity of AWQ {label} model ({arch}, {num_bits}-bit): {ppl:.2f}")

    return model_awq, fp_ppl, ppl

MODEL_ID = "unsloth/Llama-3.2-1B"
# MODEL_ID = "openai-community/gpt2"

ds = load_dataset("garage-bAInd/Open-Platypus", split="train")
calib_corpus = [ds[i]["instruction"] for i in range(256)]
eval_corpus  = [ds[i]["instruction"] for i in range(256, 512)]

model_awq, fp_ppl, awq_ppl = run_awq_quantization(
    MODEL_ID,
    calib_corpus,
    eval_corpus,
    output_dir="./awq-4bit-qkv-wo",
    include_wo=True,
    num_bits=6,
)